In [1]:
# Batch 0 / Cell 1 - Setup and path contract
from __future__ import annotations

import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "tools").is_dir():
            return candidate
        if (candidate / "notebooks").is_dir() and (candidate / "outputs").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

NOTEBOOK_ID = "25_stable_diffusion_feature_similarity"
NOTEBOOK_LABEL = "Stable Diffusion Feature Similarity"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_ID
OUTPUT_DIRS = {
    "inventory": OUTPUT_ROOT / "inventory",
    "tables": OUTPUT_ROOT / "tables",
    "metrics": OUTPUT_ROOT / "metrics",
    "analysis": OUTPUT_ROOT / "analysis",
    "figures": OUTPUT_ROOT / "figures",
    "validation": OUTPUT_ROOT / "validation",
    "manifests": OUTPUT_ROOT / "manifests",
    "reports": OUTPUT_ROOT / "reports",
}

for output_dir in OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)

# Batch 0 outputs
BATCH0_INVENTORY_SNAPSHOT_PATH = OUTPUT_DIRS["inventory"] / "batch0_project_inventory_snapshot.csv"
BATCH0_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch0_validation.csv"
STAGE_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_feature_similarity_stage_manifest.json"

# Planned downstream outputs
BATCH1_INPUT_CASES_PATH = OUTPUT_DIRS["tables"] / "stable_diffusion_feature_input_cases.csv"
BATCH1_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch1_input_validation.csv"

BATCH2_SMOKE_METRICS_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_smoke.csv"
BATCH2_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch2_smoke_validation.csv"

BATCH3_METRICS_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_metrics.csv"
BATCH3_EMBEDDINGS_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_embeddings.npz"
BATCH3_EMBEDDING_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_feature_similarity_embedding_manifest.csv"
BATCH3_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_metrics_validation.csv"

BATCH4_SUMMARY_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_summary.csv"
BATCH4_SELECTED_CASES_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_selected_cases.csv"
BATCH4_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv"

BATCH5_FIGURE_DIR = OUTPUT_DIRS["figures"] / "feature_similarity_diagnostics"
BATCH5_FIGURE_MANIFEST_PATH = OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_figure_manifest.csv"
BATCH5_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch5_figure_validation.csv"

BATCH6_ARTIFACT_INDEX_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_feature_similarity_artifact_index.csv"
BATCH6_HANDOFF_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_feature_similarity_handoff_manifest.json"
BATCH6_FINAL_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_final_validation.csv"

# Upstream Notebook 24 artifacts
UPSTREAM_NOTEBOOK_ID = "24_stable_diffusion_lpips_metrics"
UPSTREAM_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / UPSTREAM_NOTEBOOK_ID
UPSTREAM_INPUT_CASES_PATH = UPSTREAM_OUTPUT_ROOT / "tables" / "stable_diffusion_lpips_input_cases.csv"
UPSTREAM_FINAL_VALIDATION_PATH = UPSTREAM_OUTPUT_ROOT / "validation" / "stable_diffusion_lpips_metrics_final_validation.csv"
UPSTREAM_HANDOFF_MANIFEST_PATH = UPSTREAM_OUTPUT_ROOT / "manifests" / "stable_diffusion_lpips_metrics_handoff_manifest.json"

# Project inventory
PROJECT_INVENTORY_DIR = PROJECT_ROOT / "outputs" / "inventory"
PROJECT_INVENTORY_PATH = PROJECT_INVENTORY_DIR / "project_file_inventory.csv"
INVENTORY_SCRIPT_PATH = PROJECT_ROOT / "tools" / "build_project_inventory.py"

EXPECTED_CANDIDATE_ROWS = 945
EXPECTED_ZERO_CONTROL_ROWS = 94
EXPECTED_REGION_COUNTS = {
    "full_image": 945,
    "content_region": 945,
    "mask_bbox_crop": 834,
}
EXPECTED_FEATURE_ROWS = sum(EXPECTED_REGION_COUNTS.values())
EXPECTED_EMBEDDINGS_PER_FEATURE_ROW = 6
EXPECTED_EMBEDDING_ROWS = EXPECTED_FEATURE_ROWS * EXPECTED_EMBEDDINGS_PER_FEATURE_ROW

FEATURE_INPUT_SIZE = 224
TARGET_SIZE = 768
MASK_BINARY_THRESHOLD = 0
MASK_BBOX_MARGIN = 12

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Upstream input cases: {UPSTREAM_INPUT_CASES_PATH}")

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Output root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\25_stable_diffusion_feature_similarity
Upstream input cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\24_stable_diffusion_lpips_metrics\tables\stable_diffusion_lpips_input_cases.csv


In [2]:
# Batch 0 / Cell 2 - Shared helper functions
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def bool_series(series: pd.Series) -> pd.Series:
    if series.empty:
        return series.astype(bool)

    if series.dtype == bool:
        return series.fillna(False)

    truthy = {"true", "1", "yes", "y", "passed", "pass"}
    return series.fillna(False).map(
        lambda value: bool(value)
        if isinstance(value, (bool, np.bool_))
        else str(value).strip().lower() in truthy
    )


def validation_row(
    check_name: str,
    actual,
    expected,
    passed: bool,
    failure_message: str,
) -> dict:
    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
        "checked_at_utc": utc_now_iso(),
    }


def to_json_safe(value):
    if isinstance(value, Path):
        return rel(value)
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if not isinstance(value, (list, tuple, dict, set, Path)) else False:
        return None
    return value


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if Path(path).is_file() else pd.DataFrame()


def read_json_if_exists(path: Path) -> dict:
    if not Path(path).is_file():
        return {}
    return json.loads(Path(path).read_text(encoding="utf-8"))


def validation_file_passed(path: Path) -> bool:
    if not Path(path).is_file():
        return False
    validation_df = pd.read_csv(path)
    return (
        not validation_df.empty
        and "passed" in validation_df.columns
        and bool_series(validation_df["passed"]).all()
    )


def run_project_inventory() -> dict:
    PROJECT_INVENTORY_DIR.mkdir(parents=True, exist_ok=True)

    if not INVENTORY_SCRIPT_PATH.is_file():
        return {
            "passed": False,
            "returncode": None,
            "stdout": "",
            "stderr": f"Inventory script not found: {INVENTORY_SCRIPT_PATH}",
        }

    command = [
        sys.executable,
        str(INVENTORY_SCRIPT_PATH),
        "--root",
        str(PROJECT_ROOT),
        "--out-dir",
        str(PROJECT_INVENTORY_DIR),
    ]

    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )

    return {
        "passed": completed.returncode == 0,
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
        "command": " ".join(command),
    }


def planned_output_contract() -> dict:
    return {
        "batch0": {
            "inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
            "validation": rel(BATCH0_VALIDATION_PATH),
            "stage_manifest": rel(STAGE_MANIFEST_PATH),
        },
        "batch1": {
            "input_cases": rel(BATCH1_INPUT_CASES_PATH),
            "validation": rel(BATCH1_VALIDATION_PATH),
        },
        "batch2": {
            "smoke_metrics": rel(BATCH2_SMOKE_METRICS_PATH),
            "validation": rel(BATCH2_VALIDATION_PATH),
        },
        "batch3": {
            "feature_metrics": rel(BATCH3_METRICS_PATH),
            "embedding_bundle": rel(BATCH3_EMBEDDINGS_PATH),
            "embedding_manifest": rel(BATCH3_EMBEDDING_MANIFEST_PATH),
            "validation": rel(BATCH3_VALIDATION_PATH),
        },
        "batch4": {
            "summary": rel(BATCH4_SUMMARY_PATH),
            "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
            "validation": rel(BATCH4_VALIDATION_PATH),
        },
        "batch5": {
            "figure_dir": rel(BATCH5_FIGURE_DIR),
            "figure_manifest": rel(BATCH5_FIGURE_MANIFEST_PATH),
            "validation": rel(BATCH5_VALIDATION_PATH),
        },
        "batch6": {
            "artifact_index": rel(BATCH6_ARTIFACT_INDEX_PATH),
            "handoff_manifest": rel(BATCH6_HANDOFF_MANIFEST_PATH),
            "final_validation": rel(BATCH6_FINAL_VALIDATION_PATH),
        },
    }


print("Batch 0 helpers ready.")

Batch 0 helpers ready.


In [3]:
# Batch 0 / Cell 3 - Refresh inventory, validate helper API, and gate upstream artifacts
batch0_started_at_utc = utc_now_iso()
inventory_result = run_project_inventory()

inventory_df = read_csv_if_exists(PROJECT_INVENTORY_PATH)
if PROJECT_INVENTORY_PATH.is_file():
    shutil.copy2(PROJECT_INVENTORY_PATH, BATCH0_INVENTORY_SNAPSHOT_PATH)

from restoration_eval import metrics_feature_similarity as feature_similarity

helper_required_attributes = [
    "FEATURE_SIMILARITY_METRIC_SCHEMA_VERSION",
    "FEATURE_SIMILARITY_EMBEDDING_SCHEMA_VERSION",
    "FEATURE_SIMILARITY_EVALUATION_REGIONS",
    "compute_feature_similarity_for_restorations",
    "compute_feature_similarity_for_restorations_with_embeddings",
    "save_feature_embedding_bundle",
    "validate_feature_embedding_manifest",
    "expected_feature_similarity_rows_from_metadata",
    "expected_feature_similarity_region_counts_from_metadata",
]

missing_helper_attributes = [
    name for name in helper_required_attributes
    if not hasattr(feature_similarity, name)
]

feature_signature = inspect.signature(
    feature_similarity.compute_feature_similarity_for_restorations
)
with_embeddings_signature = inspect.signature(
    feature_similarity.compute_feature_similarity_for_restorations_with_embeddings
)

helper_regions = set(getattr(feature_similarity, "FEATURE_SIMILARITY_EVALUATION_REGIONS", []))
expected_regions = {"full_image", "content_region", "mask_bbox_crop"}

upstream_input_cases_df = read_csv_if_exists(UPSTREAM_INPUT_CASES_PATH)
upstream_final_validation_df = read_csv_if_exists(UPSTREAM_FINAL_VALIDATION_PATH)
upstream_handoff_manifest = read_json_if_exists(UPSTREAM_HANDOFF_MANIFEST_PATH)

upstream_zero_control_count = None
if not upstream_input_cases_df.empty and "is_zero_control" in upstream_input_cases_df.columns:
    upstream_zero_control_count = int(bool_series(upstream_input_cases_df["is_zero_control"]).sum())

upstream_handoff_status = str(upstream_handoff_manifest.get("status", "")).strip().lower()
upstream_handoff_counts = upstream_handoff_manifest.get("counts", {})

batch0_validation_rows = [
    validation_row(
        "project_root_exists",
        rel(PROJECT_ROOT),
        "directory exists",
        PROJECT_ROOT.is_dir(),
        "Project root could not be resolved.",
    ),
    validation_row(
        "output_root_created",
        rel(OUTPUT_ROOT),
        "directory exists",
        OUTPUT_ROOT.is_dir(),
        "Notebook 25 output root was not created.",
    ),
    validation_row(
        "inventory_script_exists",
        rel(INVENTORY_SCRIPT_PATH),
        "file exists",
        INVENTORY_SCRIPT_PATH.is_file(),
        "Inventory script is missing.",
    ),
    validation_row(
        "inventory_refresh_completed",
        inventory_result.get("returncode"),
        0,
        bool(inventory_result.get("passed")),
        inventory_result.get("stderr", "Inventory refresh failed."),
    ),
    validation_row(
        "project_inventory_exists",
        rel(PROJECT_INVENTORY_PATH),
        "file exists",
        PROJECT_INVENTORY_PATH.is_file(),
        "Project inventory CSV is missing after refresh.",
    ),
    validation_row(
        "project_inventory_non_empty",
        len(inventory_df),
        "> 0",
        len(inventory_df) > 0,
        "Project inventory CSV is empty.",
    ),
    validation_row(
        "batch0_inventory_snapshot_written",
        rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "file exists",
        BATCH0_INVENTORY_SNAPSHOT_PATH.is_file(),
        "Batch 0 inventory snapshot was not written.",
    ),
    validation_row(
        "helper_required_api_present",
        missing_helper_attributes,
        [],
        len(missing_helper_attributes) == 0,
        "Feature-similarity helper is missing required API attributes.",
    ),
    validation_row(
        "helper_metric_schema_version",
        getattr(feature_similarity, "FEATURE_SIMILARITY_METRIC_SCHEMA_VERSION", ""),
        ">= 2.1.0",
        str(getattr(feature_similarity, "FEATURE_SIMILARITY_METRIC_SCHEMA_VERSION", "")) >= "2.1.0",
        "Feature-similarity helper metric schema version is older than expected.",
    ),
    validation_row(
        "helper_embedding_schema_version",
        getattr(feature_similarity, "FEATURE_SIMILARITY_EMBEDDING_SCHEMA_VERSION", ""),
        "1.0.0",
        str(getattr(feature_similarity, "FEATURE_SIMILARITY_EMBEDDING_SCHEMA_VERSION", "")) == "1.0.0",
        "Feature-similarity helper embedding schema version is not the expected version.",
    ),
    validation_row(
        "helper_regions_match_contract",
        sorted(helper_regions),
        sorted(expected_regions),
        helper_regions == expected_regions,
        "Helper evaluation regions do not match the Notebook 25 contract.",
    ),
    validation_row(
        "helper_supports_return_embeddings",
        "return_embeddings" in feature_signature.parameters,
        True,
        "return_embeddings" in feature_signature.parameters,
        "compute_feature_similarity_for_restorations does not expose return_embeddings.",
    ),
    validation_row(
        "helper_embedding_wrapper_signature",
        "embedding_dtype" in with_embeddings_signature.parameters,
        True,
        "embedding_dtype" in with_embeddings_signature.parameters,
        "Embedding wrapper does not expose embedding_dtype.",
    ),
    validation_row(
        "upstream_input_cases_exists",
        rel(UPSTREAM_INPUT_CASES_PATH),
        "file exists",
        UPSTREAM_INPUT_CASES_PATH.is_file(),
        "Notebook 24 input cases CSV is missing. Run/finalize Notebook 24 first.",
    ),
    validation_row(
        "upstream_input_cases_non_empty",
        len(upstream_input_cases_df),
        "> 0",
        len(upstream_input_cases_df) > 0,
        "Notebook 24 input cases CSV is empty or unreadable.",
    ),
    validation_row(
        "upstream_candidate_row_count",
        len(upstream_input_cases_df),
        EXPECTED_CANDIDATE_ROWS,
        len(upstream_input_cases_df) == EXPECTED_CANDIDATE_ROWS,
        "Notebook 24 input case count does not match the expected 945-case contract.",
    ),
    validation_row(
        "upstream_zero_control_count",
        upstream_zero_control_count,
        EXPECTED_ZERO_CONTROL_ROWS,
        upstream_zero_control_count == EXPECTED_ZERO_CONTROL_ROWS,
        "Notebook 24 zero-control count does not match the expected 94-case contract.",
    ),
    validation_row(
        "upstream_final_validation_exists",
        rel(UPSTREAM_FINAL_VALIDATION_PATH),
        "file exists",
        UPSTREAM_FINAL_VALIDATION_PATH.is_file(),
        "Notebook 24 final validation CSV is missing.",
    ),
    validation_row(
        "upstream_final_validation_passed",
        validation_file_passed(UPSTREAM_FINAL_VALIDATION_PATH),
        True,
        validation_file_passed(UPSTREAM_FINAL_VALIDATION_PATH),
        "Notebook 24 final validation did not pass.",
    ),
    validation_row(
        "upstream_handoff_manifest_exists",
        rel(UPSTREAM_HANDOFF_MANIFEST_PATH),
        "file exists",
        UPSTREAM_HANDOFF_MANIFEST_PATH.is_file(),
        "Notebook 24 handoff manifest JSON is missing.",
    ),
    validation_row(
        "upstream_handoff_status_passed",
        upstream_handoff_status,
        "passed",
        upstream_handoff_status == "passed",
        "Notebook 24 handoff manifest status is not passed.",
    ),
    validation_row(
        "expected_feature_row_contract",
        EXPECTED_FEATURE_ROWS,
        2724,
        EXPECTED_FEATURE_ROWS == 2724,
        "Notebook 25 expected feature-row contract is wrong.",
    ),
    validation_row(
        "expected_embedding_row_contract",
        EXPECTED_EMBEDDING_ROWS,
        16344,
        EXPECTED_EMBEDDING_ROWS == 16344,
        "Notebook 25 expected embedding-row contract is wrong.",
    ),
]

batch0_validation_df = pd.DataFrame(batch0_validation_rows)
batch0_validation_df.to_csv(BATCH0_VALIDATION_PATH, index=False)

batch0_passed = bool_series(batch0_validation_df["passed"]).all()

print(f"Inventory refresh return code: {inventory_result.get('returncode')}")
print(f"Inventory rows: {len(inventory_df):,}")
print(f"Helper metric schema: {getattr(feature_similarity, 'FEATURE_SIMILARITY_METRIC_SCHEMA_VERSION', '')}")
print(f"Helper embedding schema: {getattr(feature_similarity, 'FEATURE_SIMILARITY_EMBEDDING_SCHEMA_VERSION', '')}")
print(f"Upstream input rows: {len(upstream_input_cases_df):,}")
print(f"Batch 0 checks passed: {int(bool_series(batch0_validation_df['passed']).sum())} / {len(batch0_validation_df)}")

display(batch0_validation_df)

Inventory refresh return code: 0
Inventory rows: 10,831
Helper metric schema: 2.1.0
Helper embedding schema: 1.0.0
Upstream input rows: 945
Batch 0 checks passed: 23 / 23


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,project_root_exists,.,directory exists,True,,2026-08-08T17:18:00+00:00
1,output_root_created,outputs/25_stable_diffusion_feature_similarity,directory exists,True,,2026-08-08T17:18:00+00:00
2,inventory_script_exists,tools/build_project_inventory.py,file exists,True,,2026-08-08T17:18:00+00:00
3,inventory_refresh_completed,0,0,True,,2026-08-08T17:18:00+00:00
4,project_inventory_exists,outputs/inventory/project_file_inventory.csv,file exists,True,,2026-08-08T17:18:00+00:00
5,project_inventory_non_empty,10831,> 0,True,,2026-08-08T17:18:00+00:00
6,batch0_inventory_snapshot_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T17:18:00+00:00
7,helper_required_api_present,[],[],True,,2026-08-08T17:18:00+00:00
8,helper_metric_schema_version,2.1.0,>= 2.1.0,True,,2026-08-08T17:18:00+00:00
9,helper_embedding_schema_version,1.0.0,1.0.0,True,,2026-08-08T17:18:00+00:00


In [4]:
# Batch 0 / Cell 4 - Write stage manifest and enforce Batch 0 gate
stage_manifest = {
    "schema_version": "stable_diffusion_feature_similarity_stage_manifest_v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_label": NOTEBOOK_LABEL,
    "stage": "batch0_setup_inventory_helper_upstream_contract",
    "stage_status": "passed" if batch0_passed else "failed",
    "created_at_utc": batch0_started_at_utc,
    "updated_at_utc": utc_now_iso(),
    "project_root": rel(PROJECT_ROOT),
    "output_root": rel(OUTPUT_ROOT),
    "upstream": {
        "notebook_id": UPSTREAM_NOTEBOOK_ID,
        "input_cases": rel(UPSTREAM_INPUT_CASES_PATH),
        "final_validation": rel(UPSTREAM_FINAL_VALIDATION_PATH),
        "handoff_manifest": rel(UPSTREAM_HANDOFF_MANIFEST_PATH),
        "handoff_status": upstream_handoff_status,
        "handoff_counts": upstream_handoff_counts,
    },
    "helper": {
        "module": "restoration_eval.metrics_feature_similarity",
        "metric_schema_version": getattr(feature_similarity, "FEATURE_SIMILARITY_METRIC_SCHEMA_VERSION", ""),
        "embedding_schema_version": getattr(feature_similarity, "FEATURE_SIMILARITY_EMBEDDING_SCHEMA_VERSION", ""),
        "evaluation_regions": sorted(helper_regions),
        "supports_retained_embeddings": (
            hasattr(feature_similarity, "compute_feature_similarity_for_restorations_with_embeddings")
            and "return_embeddings" in feature_signature.parameters
        ),
    },
    "contracts": {
        "expected_candidate_rows": EXPECTED_CANDIDATE_ROWS,
        "expected_zero_control_rows": EXPECTED_ZERO_CONTROL_ROWS,
        "expected_region_counts": EXPECTED_REGION_COUNTS,
        "expected_feature_rows": EXPECTED_FEATURE_ROWS,
        "expected_embeddings_per_feature_row": EXPECTED_EMBEDDINGS_PER_FEATURE_ROW,
        "expected_embedding_rows": EXPECTED_EMBEDDING_ROWS,
        "target_size": TARGET_SIZE,
        "feature_input_size": FEATURE_INPUT_SIZE,
        "mask_binary_threshold": MASK_BINARY_THRESHOLD,
        "mask_bbox_margin": MASK_BBOX_MARGIN,
        "feature_improvement_direction": "restored_similarity - damaged_similarity; higher is better",
        "embedding_storage_policy": "vectors stored in compressed NPZ; CSV stores only manifest keys and metadata",
    },
    "outputs": planned_output_contract(),
    "batch0": {
        "validation": rel(BATCH0_VALIDATION_PATH),
        "inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "checks_passed": int(bool_series(batch0_validation_df["passed"]).sum()),
        "checks_total": int(len(batch0_validation_df)),
        "passed": bool(batch0_passed),
    },
    "runtime": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "executable": sys.executable,
    },
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved validation: {rel(BATCH0_VALIDATION_PATH)}")
print(f"Saved stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Saved inventory snapshot: {rel(BATCH0_INVENTORY_SNAPSHOT_PATH)}")

if not batch0_passed:
    display(batch0_validation_df.loc[~bool_series(batch0_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 0 failed. Fix the failed checks before continuing to Batch 1.")

print("Batch 0 passed. Continue to Batch 1.")

Saved validation: outputs/25_stable_diffusion_feature_similarity/validation/batch0_validation.csv
Saved stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Saved inventory snapshot: outputs/25_stable_diffusion_feature_similarity/inventory/batch0_project_inventory_snapshot.csv
Batch 0 passed. Continue to Batch 1.


In [5]:
# Batch 1 / Cell 1 - Input validation helpers
from PIL import Image

from restoration_eval import metrics_feature_similarity as feature_similarity

FEATURE_MASK_BINARY_THRESHOLD = int(
    getattr(feature_similarity, "DEFAULT_MASK_BINARY_THRESHOLD", 127)
)
FEATURE_MASK_BBOX_MARGIN = int(
    getattr(feature_similarity, "DEFAULT_MASK_BBOX_MARGIN", 8)
)

REQUIRED_FEATURE_INPUT_COLUMNS = [
    "case_id",
    "candidate_id",
    "restoration_case_id",
    "painting_id",
    "category",
    "title",
    "mask_id",
    "mask_type",
    "model_name",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

RECOMMENDED_FEATURE_INPUT_COLUMNS = [
    "candidate_index",
    "candidate_seed",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "prompt_variant_family",
    "effective_candidate_seed",
    "inference_mode",
    "execution_device",
    "scheduler_name",
    "num_inference_steps",
    "guidance_scale",
    "strength",
    "hf_model_id",
    "model_revision",
    "is_zero_control",
]

PATH_COLUMNS = ["clean_path", "damaged_path", "restored_path", "mask_path"]
IMAGE_PATH_COLUMNS = ["clean_path", "damaged_path", "restored_path"]


def resolve_project_path(path_value) -> Path:
    path_text = str(path_value).strip()
    path_text = path_text.replace("\\", "/")
    path = Path(path_text)

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def inspect_image_header(path_value) -> dict:
    resolved_path = resolve_project_path(path_value)
    result = {
        "path": str(path_value),
        "resolved_path": resolved_path,
        "exists": resolved_path.is_file(),
        "non_empty": False,
        "readable": False,
        "width": np.nan,
        "height": np.nan,
        "mode": "",
        "error": "",
    }

    if not result["exists"]:
        result["error"] = "file missing"
        return result

    try:
        result["non_empty"] = resolved_path.stat().st_size > 0
        with Image.open(resolved_path) as image:
            result["width"] = int(image.width)
            result["height"] = int(image.height)
            result["mode"] = str(image.mode)
            result["readable"] = True
    except Exception as exc:
        result["error"] = repr(exc)

    return result


def inspect_mask(path_value, threshold: int = FEATURE_MASK_BINARY_THRESHOLD) -> dict:
    resolved_path = resolve_project_path(path_value)
    result = inspect_image_header(path_value)
    result.update(
        {
            "mask_has_positive_pixels": False,
            "mask_positive_pixel_count": 0,
            "mask_total_pixel_count": 0,
        }
    )

    if not result["readable"]:
        return result

    try:
        with Image.open(resolved_path) as image:
            mask_arr = np.asarray(image.convert("L"))
        mask_bool = mask_arr > int(threshold)
        result["mask_has_positive_pixels"] = bool(mask_bool.any())
        result["mask_positive_pixel_count"] = int(mask_bool.sum())
        result["mask_total_pixel_count"] = int(mask_bool.size)
    except Exception as exc:
        result["readable"] = False
        result["error"] = repr(exc)

    return result


def infer_zero_control_series(df: pd.DataFrame) -> pd.Series:
    if "is_zero_control" in df.columns:
        return bool_series(df["is_zero_control"])

    zero_markers = pd.Series(False, index=df.index)

    for column in ["case_id", "candidate_id", "restoration_case_id", "mask_type", "mask_id"]:
        if column in df.columns:
            zero_markers = zero_markers | df[column].astype(str).str.lower().str.contains(
                "zero", na=False
            )

    return zero_markers.fillna(False)


def make_expected_regions(mask_has_positive_pixels: bool) -> str:
    regions = ["full_image", "content_region"]
    if bool(mask_has_positive_pixels):
        regions.append("mask_bbox_crop")
    return "|".join(regions)


print(f"Feature mask threshold: {FEATURE_MASK_BINARY_THRESHOLD}")
print(f"Feature mask bbox margin: {FEATURE_MASK_BBOX_MARGIN}")

Feature mask threshold: 127
Feature mask bbox margin: 8


In [7]:
# Batch 1 / Cell 2 - Load upstream input cases and audit files/content boxes
batch1_started_at_utc = utc_now_iso()

upstream_input_cases_df = pd.read_csv(UPSTREAM_INPUT_CASES_PATH)
feature_input_cases_df = upstream_input_cases_df.copy()

missing_required_feature_columns = [
    column for column in REQUIRED_FEATURE_INPUT_COLUMNS
    if column not in feature_input_cases_df.columns
]
missing_recommended_feature_columns = [
    column for column in RECOMMENDED_FEATURE_INPUT_COLUMNS
    if column not in feature_input_cases_df.columns
]

if missing_required_feature_columns:
    raise ValueError(
        f"Cannot build Batch 1 input cases. Missing columns: {missing_required_feature_columns}"
    )

# Remove any stale derived columns from prior notebooks/reruns so pandas cannot suffix them.
DERIVED_BATCH1_COLUMNS = [
    "feature_input_row_id",
    "source_notebook_id",
    "source_input_cases_path",
    "is_zero_control_for_feature_similarity",
    "mask_has_positive_pixels_for_feature_similarity",
    "mask_positive_pixel_count_for_feature_similarity",
    "expected_feature_regions",
    "expected_feature_region_count",
    "clean_width",
    "clean_height",
    "content_bbox_width",
    "content_bbox_height",
    "content_bbox_area",
    "content_bbox_valid_for_feature_similarity",
    "width_clean",
    "width_damaged",
    "width_restored",
    "width_mask",
    "height_clean",
    "height_damaged",
    "height_restored",
    "height_mask",
    "rgb_dimensions_match",
    "mask_dimensions_match_clean",
    "all_required_files_exist",
    "all_required_files_non_empty",
    "all_required_files_readable",
    "valid_for_feature_similarity",
]
feature_input_cases_df = feature_input_cases_df.drop(
    columns=[column for column in DERIVED_BATCH1_COLUMNS if column in feature_input_cases_df.columns],
    errors="ignore",
)

feature_input_cases_df.insert(
    0,
    "feature_input_row_id",
    np.arange(len(feature_input_cases_df), dtype=int),
)
feature_input_cases_df["source_notebook_id"] = UPSTREAM_NOTEBOOK_ID
feature_input_cases_df["source_input_cases_path"] = rel(UPSTREAM_INPUT_CASES_PATH)

zero_control_series = infer_zero_control_series(feature_input_cases_df)
feature_input_cases_df["is_zero_control_for_feature_similarity"] = zero_control_series.astype(bool)

image_audit_rows = []

for _, row in feature_input_cases_df.iterrows():
    role_inspections = {}

    for path_column in IMAGE_PATH_COLUMNS:
        role = path_column.replace("_path", "")
        role_inspections[role] = inspect_image_header(row[path_column])

    role_inspections["mask"] = inspect_mask(row["mask_path"])

    for role, inspection in role_inspections.items():
        image_audit_rows.append(
            {
                "feature_input_row_id": int(row["feature_input_row_id"]),
                "case_id": row["case_id"],
                "role": role,
                "path": inspection["path"],
                "resolved_path": str(inspection["resolved_path"]),
                "exists": bool(inspection["exists"]),
                "non_empty": bool(inspection["non_empty"]),
                "readable": bool(inspection["readable"]),
                "width": inspection["width"],
                "height": inspection["height"],
                "mode": inspection["mode"],
                "error": inspection["error"],
                "mask_has_positive_pixels": inspection.get("mask_has_positive_pixels", np.nan),
                "mask_positive_pixel_count": inspection.get("mask_positive_pixel_count", np.nan),
                "mask_total_pixel_count": inspection.get("mask_total_pixel_count", np.nan),
            }
        )

image_audit_df = pd.DataFrame(image_audit_rows)

mask_audit_df = image_audit_df.loc[image_audit_df["role"].eq("mask")].copy()

mask_positive_by_row = (
    mask_audit_df
    .set_index("feature_input_row_id")["mask_has_positive_pixels"]
    .astype(bool)
)
mask_positive_pixel_count_by_row = (
    mask_audit_df
    .set_index("feature_input_row_id")["mask_positive_pixel_count"]
    .fillna(0)
    .astype(int)
)

feature_input_cases_df["mask_has_positive_pixels_for_feature_similarity"] = (
    feature_input_cases_df["feature_input_row_id"]
    .map(mask_positive_by_row)
    .fillna(False)
    .astype(bool)
)
feature_input_cases_df["mask_positive_pixel_count_for_feature_similarity"] = (
    feature_input_cases_df["feature_input_row_id"]
    .map(mask_positive_pixel_count_by_row)
    .fillna(0)
    .astype(int)
)

feature_input_cases_df["expected_feature_regions"] = feature_input_cases_df[
    "mask_has_positive_pixels_for_feature_similarity"
].map(make_expected_regions)
feature_input_cases_df["expected_feature_region_count"] = (
    feature_input_cases_df["expected_feature_regions"].str.split("|").map(len)
)

for bbox_column in ["content_x_min", "content_y_min", "content_x_max", "content_y_max"]:
    feature_input_cases_df[bbox_column] = pd.to_numeric(
        feature_input_cases_df[bbox_column],
        errors="coerce",
    )

# Assign image dimensions directly by role, avoiding merge suffix bugs.
for role in ["clean", "damaged", "restored", "mask"]:
    role_audit_df = image_audit_df.loc[image_audit_df["role"].eq(role)].copy()
    width_by_row = role_audit_df.set_index("feature_input_row_id")["width"]
    height_by_row = role_audit_df.set_index("feature_input_row_id")["height"]

    feature_input_cases_df[f"width_{role}"] = feature_input_cases_df["feature_input_row_id"].map(width_by_row)
    feature_input_cases_df[f"height_{role}"] = feature_input_cases_df["feature_input_row_id"].map(height_by_row)

feature_input_cases_df["clean_width"] = feature_input_cases_df["width_clean"]
feature_input_cases_df["clean_height"] = feature_input_cases_df["height_clean"]

feature_input_cases_df["content_bbox_width"] = (
    feature_input_cases_df["content_x_max"] - feature_input_cases_df["content_x_min"]
)
feature_input_cases_df["content_bbox_height"] = (
    feature_input_cases_df["content_y_max"] - feature_input_cases_df["content_y_min"]
)
feature_input_cases_df["content_bbox_area"] = (
    feature_input_cases_df["content_bbox_width"] * feature_input_cases_df["content_bbox_height"]
)

feature_input_cases_df["content_bbox_valid_for_feature_similarity"] = (
    feature_input_cases_df[["content_x_min", "content_y_min", "content_x_max", "content_y_max"]].notna().all(axis=1)
    & feature_input_cases_df["content_bbox_width"].gt(0)
    & feature_input_cases_df["content_bbox_height"].gt(0)
    & feature_input_cases_df["content_x_min"].ge(0)
    & feature_input_cases_df["content_y_min"].ge(0)
    & feature_input_cases_df["content_x_max"].le(feature_input_cases_df["clean_width"])
    & feature_input_cases_df["content_y_max"].le(feature_input_cases_df["clean_height"])
)

feature_input_cases_df["rgb_dimensions_match"] = (
    feature_input_cases_df["width_clean"].eq(feature_input_cases_df["width_damaged"])
    & feature_input_cases_df["width_clean"].eq(feature_input_cases_df["width_restored"])
    & feature_input_cases_df["height_clean"].eq(feature_input_cases_df["height_damaged"])
    & feature_input_cases_df["height_clean"].eq(feature_input_cases_df["height_restored"])
)

feature_input_cases_df["mask_dimensions_match_clean"] = (
    feature_input_cases_df["width_clean"].eq(feature_input_cases_df["width_mask"])
    & feature_input_cases_df["height_clean"].eq(feature_input_cases_df["height_mask"])
)

case_file_status_df = (
    image_audit_df
    .groupby("feature_input_row_id")
    .agg(
        all_required_files_exist=("exists", "all"),
        all_required_files_non_empty=("non_empty", "all"),
        all_required_files_readable=("readable", "all"),
    )
)

for status_column in [
    "all_required_files_exist",
    "all_required_files_non_empty",
    "all_required_files_readable",
]:
    feature_input_cases_df[status_column] = (
        feature_input_cases_df["feature_input_row_id"]
        .map(case_file_status_df[status_column])
        .fillna(False)
        .astype(bool)
    )

feature_input_cases_df["valid_for_feature_similarity"] = (
    feature_input_cases_df["all_required_files_exist"]
    & feature_input_cases_df["all_required_files_non_empty"]
    & feature_input_cases_df["all_required_files_readable"]
    & feature_input_cases_df["content_bbox_valid_for_feature_similarity"].astype(bool)
    & feature_input_cases_df["rgb_dimensions_match"].astype(bool)
    & feature_input_cases_df["mask_dimensions_match_clean"].astype(bool)
)

feature_input_cases_df.to_csv(BATCH1_INPUT_CASES_PATH, index=False)

print(f"Loaded upstream cases: {len(upstream_input_cases_df):,}")
print(f"Saved feature input cases: {rel(BATCH1_INPUT_CASES_PATH)}")
print(f"Valid feature cases: {int(feature_input_cases_df['valid_for_feature_similarity'].sum()):,} / {len(feature_input_cases_df):,}")

display(
    feature_input_cases_df[
        [
            "feature_input_row_id",
            "case_id",
            "candidate_id",
            "is_zero_control_for_feature_similarity",
            "mask_has_positive_pixels_for_feature_similarity",
            "expected_feature_regions",
            "valid_for_feature_similarity",
        ]
    ].head(10)
)

Loaded upstream cases: 945
Saved feature input cases: outputs/25_stable_diffusion_feature_similarity/tables/stable_diffusion_feature_input_cases.csv
Valid feature cases: 945 / 945


,feature_input_row_id,case_id,candidate_id,is_zero_control_for_feature_similarity,mask_has_positive_pixels_for_feature_similarity,expected_feature_regions,valid_for_feature_similarity
0,0,p001_loss_large,sd__can__p001__loss_large__p00_generic__s2026_...,False,True,full_image|content_region|mask_bbox_crop,True
1,1,p001_loss_small,sd__can__p001__loss_small__p00_generic__s2026_...,False,True,full_image|content_region|mask_bbox_crop,True
2,2,p001_mixed_damage,sd__can__p001__mixed_damage__p00_generic__s202...,False,True,full_image|content_region|mask_bbox_crop,True
3,3,p001_scratch_thin,sd__can__p001__scratch_thin__p00_generic__s202...,False,True,full_image|content_region|mask_bbox_crop,True
4,4,p001_zero_control,sd__can__p001__zero__p00_generic__s2026__3be10...,True,False,full_image|content_region,True
5,5,p002_loss_large,sd__can__p002__loss_large__p00_generic__s2026_...,False,True,full_image|content_region|mask_bbox_crop,True
6,6,p002_loss_small,sd__can__p002__loss_small__p00_generic__s2026_...,False,True,full_image|content_region|mask_bbox_crop,True
7,7,p002_mixed_damage,sd__can__p002__mixed_damage__p00_generic__s202...,False,True,full_image|content_region|mask_bbox_crop,True
8,8,p002_scratch_thin,sd__can__p002__scratch_thin__p00_generic__s202...,False,True,full_image|content_region|mask_bbox_crop,True
9,9,p002_zero_control,sd__can__p002__zero__p00_generic__s2026__606dc...,True,False,full_image|content_region,True


In [8]:
# Batch 1 / Cell 3 - Validate expected feature-region contract
expected_regions_exploded = (
    feature_input_cases_df[["feature_input_row_id", "case_id", "expected_feature_regions"]]
    .assign(evaluation_region=lambda df: df["expected_feature_regions"].str.split("|"))
    .explode("evaluation_region")
)

observed_region_counts = (
    expected_regions_exploded["evaluation_region"]
    .value_counts()
    .reindex(list(feature_similarity.FEATURE_SIMILARITY_EVALUATION_REGIONS), fill_value=0)
    .astype(int)
    .to_dict()
)

observed_feature_rows = int(sum(observed_region_counts.values()))
observed_zero_control_rows = int(feature_input_cases_df["is_zero_control_for_feature_similarity"].sum())
observed_positive_mask_rows = int(
    feature_input_cases_df["mask_has_positive_pixels_for_feature_similarity"].sum()
)
observed_empty_mask_rows = int(
    (~feature_input_cases_df["mask_has_positive_pixels_for_feature_similarity"]).sum()
)
observed_empty_nonzero_mask_rows = int(
    (
        ~feature_input_cases_df["is_zero_control_for_feature_similarity"]
        & ~feature_input_cases_df["mask_has_positive_pixels_for_feature_similarity"]
    ).sum()
)

helper_metadata_for_expected_counts_df = feature_input_cases_df.copy()
for path_column in PATH_COLUMNS:
    helper_metadata_for_expected_counts_df[path_column] = helper_metadata_for_expected_counts_df[
        path_column
    ].map(lambda value: str(resolve_project_path(value)))

helper_expected_region_counts = feature_similarity.expected_feature_similarity_region_counts_from_metadata(
    helper_metadata_for_expected_counts_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)
helper_expected_feature_rows = feature_similarity.expected_feature_similarity_rows_from_metadata(
    helper_metadata_for_expected_counts_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)

path_audit_summary_df = (
    image_audit_df
    .groupby("role")
    .agg(
        rows=("role", "size"),
        missing=("exists", lambda values: int((~values.astype(bool)).sum())),
        empty=("non_empty", lambda values: int((~values.astype(bool)).sum())),
        unreadable=("readable", lambda values: int((~values.astype(bool)).sum())),
        min_width=("width", "min"),
        max_width=("width", "max"),
        min_height=("height", "min"),
        max_height=("height", "max"),
    )
    .reset_index()
)

region_contract_df = pd.DataFrame(
    [
        {
            "evaluation_region": region,
            "observed_rows": int(observed_region_counts.get(region, 0)),
            "expected_rows": int(EXPECTED_REGION_COUNTS.get(region, 0)),
            "helper_expected_rows": int(helper_expected_region_counts.get(region, 0)),
        }
        for region in feature_similarity.FEATURE_SIMILARITY_EVALUATION_REGIONS
    ]
)
region_contract_df["matches_contract"] = (
    region_contract_df["observed_rows"].eq(region_contract_df["expected_rows"])
    & region_contract_df["observed_rows"].eq(region_contract_df["helper_expected_rows"])
)

print("Path audit summary:")
display(path_audit_summary_df)

print("Region contract:")
display(region_contract_df)

print(f"Observed feature rows: {observed_feature_rows:,}")
print(f"Observed zero controls: {observed_zero_control_rows:,}")
print(f"Observed empty nonzero-mask rows: {observed_empty_nonzero_mask_rows:,}")

Path audit summary:


,role,rows,missing,empty,unreadable,min_width,max_width,min_height,max_height
0,clean,945,0,0,0,768,768,768,768
1,damaged,945,0,0,0,768,768,768,768
2,mask,945,0,0,0,768,768,768,768
3,restored,945,0,0,0,768,768,768,768


Region contract:


,evaluation_region,observed_rows,expected_rows,helper_expected_rows,matches_contract
0,full_image,945,945,945,True
1,content_region,945,945,945,True
2,mask_bbox_crop,834,834,834,True


Observed feature rows: 2,724
Observed zero controls: 94
Observed empty nonzero-mask rows: 17


In [10]:
# Batch 1 / Cell 4 - Batch 1 validation and stage manifest update
missing_file_rows = int((~image_audit_df["exists"].astype(bool)).sum())
empty_file_rows = int((~image_audit_df["non_empty"].astype(bool)).sum())
unreadable_file_rows = int((~image_audit_df["readable"].astype(bool)).sum())

invalid_content_bbox_rows = int(
    (~feature_input_cases_df["content_bbox_valid_for_feature_similarity"].astype(bool)).sum()
)
rgb_dimension_mismatch_rows = int(
    (~feature_input_cases_df["rgb_dimensions_match"].astype(bool)).sum()
)
mask_dimension_mismatch_rows = int(
    (~feature_input_cases_df["mask_dimensions_match_clean"].astype(bool)).sum()
)
invalid_feature_case_rows = int(
    (~feature_input_cases_df["valid_for_feature_similarity"].astype(bool)).sum()
)

duplicate_case_id_count = int(feature_input_cases_df["case_id"].duplicated().sum())
duplicate_candidate_id_count = int(feature_input_cases_df["candidate_id"].duplicated().sum())
duplicate_restoration_case_id_count = int(
    feature_input_cases_df["restoration_case_id"].duplicated().sum()
)

batch1_validation_rows = [
    validation_row(
        "batch0_validation_passed",
        validation_file_passed(BATCH0_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH0_VALIDATION_PATH),
        "Batch 0 validation did not pass.",
    ),
    validation_row(
        "upstream_input_cases_loaded",
        len(upstream_input_cases_df),
        EXPECTED_CANDIDATE_ROWS,
        len(upstream_input_cases_df) == EXPECTED_CANDIDATE_ROWS,
        "Notebook 24 input cases did not load with the expected row count.",
    ),
    validation_row(
        "required_columns_present",
        missing_required_feature_columns,
        [],
        len(missing_required_feature_columns) == 0,
        "Notebook 24 input cases are missing required feature-similarity columns.",
    ),
    validation_row(
        "recommended_columns_reported",
        missing_recommended_feature_columns,
        "reported only",
        True,
        "Recommended provenance columns are reported but do not block feature computation.",
    ),
    validation_row(
        "case_id_present_and_groupable",
        {
            "missing_case_id_rows": int(feature_input_cases_df["case_id"].isna().sum()),
            "unique_case_ids": int(feature_input_cases_df["case_id"].nunique()),
            "duplicate_case_id_rows": duplicate_case_id_count,
        },
        "case_id may repeat across candidates; candidate_id/restoration_case_id must be unique",
        feature_input_cases_df["case_id"].notna().all(),
        "One or more case_id values is missing.",
    ),
    validation_row(
        "candidate_id_unique",
        duplicate_candidate_id_count,
        0,
        duplicate_candidate_id_count == 0,
        "Duplicate candidate_id values found.",
    ),
    validation_row(
        "restoration_case_id_unique",
        duplicate_restoration_case_id_count,
        0,
        duplicate_restoration_case_id_count == 0,
        "Duplicate restoration_case_id values found.",
    ),
    validation_row(
        "zero_control_count",
        observed_zero_control_rows,
        EXPECTED_ZERO_CONTROL_ROWS,
        observed_zero_control_rows == EXPECTED_ZERO_CONTROL_ROWS,
        "Zero-control count does not match the Notebook 24 contract.",
    ),
    validation_row(
        "positive_mask_case_count",
        observed_positive_mask_rows,
        EXPECTED_REGION_COUNTS["mask_bbox_crop"],
        observed_positive_mask_rows == EXPECTED_REGION_COUNTS["mask_bbox_crop"],
        "Positive-mask case count does not match expected mask_bbox_crop rows.",
    ),
    validation_row(
        "empty_mask_case_count",
        observed_empty_mask_rows,
        EXPECTED_CANDIDATE_ROWS - EXPECTED_REGION_COUNTS["mask_bbox_crop"],
        observed_empty_mask_rows == EXPECTED_CANDIDATE_ROWS - EXPECTED_REGION_COUNTS["mask_bbox_crop"],
        "Empty-mask case count does not match expected non-mask-bbox rows.",
    ),
    validation_row(
        "empty_nonzero_mask_case_count",
        observed_empty_nonzero_mask_rows,
        17,
        observed_empty_nonzero_mask_rows == 17,
        "Expected 17 non-zero-control cases with empty observed masks.",
    ),
    validation_row(
        "all_required_files_exist",
        missing_file_rows,
        0,
        missing_file_rows == 0,
        "One or more clean/damaged/restored/mask files is missing.",
    ),
    validation_row(
        "all_required_files_non_empty",
        empty_file_rows,
        0,
        empty_file_rows == 0,
        "One or more clean/damaged/restored/mask files is empty.",
    ),
    validation_row(
        "all_required_files_readable",
        unreadable_file_rows,
        0,
        unreadable_file_rows == 0,
        "One or more clean/damaged/restored/mask files could not be opened.",
    ),
    validation_row(
        "content_boxes_valid",
        invalid_content_bbox_rows,
        0,
        invalid_content_bbox_rows == 0,
        "One or more content bounding boxes is invalid or outside image bounds.",
    ),
    validation_row(
        "rgb_dimensions_match",
        rgb_dimension_mismatch_rows,
        0,
        rgb_dimension_mismatch_rows == 0,
        "One or more clean/damaged/restored image dimension sets does not match.",
    ),
    validation_row(
        "mask_dimensions_match_clean",
        mask_dimension_mismatch_rows,
        0,
        mask_dimension_mismatch_rows == 0,
        "One or more masks does not match the clean image dimensions.",
    ),
    validation_row(
        "all_cases_valid_for_feature_similarity",
        invalid_feature_case_rows,
        0,
        invalid_feature_case_rows == 0,
        "One or more cases is not valid for feature similarity.",
    ),
    validation_row(
        "region_counts_match_contract",
        observed_region_counts,
        EXPECTED_REGION_COUNTS,
        observed_region_counts == EXPECTED_REGION_COUNTS,
        "Expected feature-region counts do not match the Notebook 25 contract.",
    ),
    validation_row(
        "helper_region_counts_match_observed",
        helper_expected_region_counts,
        observed_region_counts,
        helper_expected_region_counts == observed_region_counts,
        "Helper-computed expected region counts do not match Batch 1 observed counts.",
    ),
    validation_row(
        "expected_feature_rows_match_contract",
        observed_feature_rows,
        EXPECTED_FEATURE_ROWS,
        observed_feature_rows == EXPECTED_FEATURE_ROWS,
        "Expected feature row count does not match 2,724-row contract.",
    ),
    validation_row(
        "helper_expected_feature_rows_match_contract",
        helper_expected_feature_rows,
        EXPECTED_FEATURE_ROWS,
        helper_expected_feature_rows == EXPECTED_FEATURE_ROWS,
        "Helper-computed expected feature row count does not match 2,724-row contract.",
    ),
    validation_row(
        "feature_input_cases_written",
        rel(BATCH1_INPUT_CASES_PATH),
        "file exists",
        BATCH1_INPUT_CASES_PATH.is_file(),
        "Batch 1 feature input cases CSV was not written.",
    ),
]

batch1_validation_df = pd.DataFrame(batch1_validation_rows)
batch1_validation_df.to_csv(BATCH1_VALIDATION_PATH, index=False)

batch1_passed = bool_series(batch1_validation_df["passed"]).all()

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch1_feature_input_cases_validation"
stage_manifest["stage_status"] = "passed" if batch1_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest.setdefault("contracts", {})
stage_manifest["contracts"]["mask_binary_threshold"] = FEATURE_MASK_BINARY_THRESHOLD
stage_manifest["contracts"]["mask_bbox_margin"] = FEATURE_MASK_BBOX_MARGIN
stage_manifest["contracts"]["expected_region_counts"] = EXPECTED_REGION_COUNTS
stage_manifest["contracts"]["expected_feature_rows"] = EXPECTED_FEATURE_ROWS
stage_manifest["batch1"] = {
    "status": "passed" if batch1_passed else "failed",
    "validation": rel(BATCH1_VALIDATION_PATH),
    "input_cases": rel(BATCH1_INPUT_CASES_PATH),
    "source_input_cases": rel(UPSTREAM_INPUT_CASES_PATH),
    "rows": int(len(feature_input_cases_df)),
    "valid_rows": int(feature_input_cases_df["valid_for_feature_similarity"].sum()),
    "zero_control_rows": observed_zero_control_rows,
    "positive_mask_rows": observed_positive_mask_rows,
    "empty_mask_rows": observed_empty_mask_rows,
    "empty_nonzero_mask_rows": observed_empty_nonzero_mask_rows,
    "expected_region_counts": observed_region_counts,
    "expected_feature_rows": observed_feature_rows,
    "checks_passed": int(bool_series(batch1_validation_df["passed"]).sum()),
    "checks_total": int(len(batch1_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved input cases: {rel(BATCH1_INPUT_CASES_PATH)}")
print(f"Saved validation: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 1 checks passed: {int(bool_series(batch1_validation_df['passed']).sum())} / {len(batch1_validation_df)}")

display(batch1_validation_df)

if not batch1_passed:
    display(batch1_validation_df.loc[~bool_series(batch1_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 1 validation failed. Fix input/path/content-box issues before Batch 2.")

print("Batch 1 passed. Continue to Batch 2.")

Saved input cases: outputs/25_stable_diffusion_feature_similarity/tables/stable_diffusion_feature_input_cases.csv
Saved validation: outputs/25_stable_diffusion_feature_similarity/validation/batch1_input_validation.csv
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 1 checks passed: 23 / 23


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch0_validation_passed,True,True,True,,2026-08-08T17:35:46+00:00
1,upstream_input_cases_loaded,945,945,True,,2026-08-08T17:35:46+00:00
2,required_columns_present,[],[],True,,2026-08-08T17:35:46+00:00
3,recommended_columns_reported,[],reported only,True,,2026-08-08T17:35:46+00:00
4,case_id_present_and_groupable,"{'missing_case_id_rows': 0, 'unique_case_ids':...",case_id may repeat across candidates; candidat...,True,,2026-08-08T17:35:46+00:00
5,candidate_id_unique,0,0,True,,2026-08-08T17:35:46+00:00
6,restoration_case_id_unique,0,0,True,,2026-08-08T17:35:46+00:00
7,zero_control_count,94,94,True,,2026-08-08T17:35:46+00:00
8,positive_mask_case_count,834,834,True,,2026-08-08T17:35:46+00:00
9,empty_mask_case_count,111,111,True,,2026-08-08T17:35:46+00:00


Batch 1 passed. Continue to Batch 2.


In [12]:
# Batch 2 / Cell 1 - Paths, runtime checks, and CLIP/DINOv2 model load
from time import perf_counter
import random
import importlib

BATCH2_SMOKE_METRICS_PATH = globals().get(
    "BATCH2_SMOKE_METRICS_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_smoke.csv",
)
BATCH2_VALIDATION_PATH = globals().get(
    "BATCH2_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch2_smoke_validation.csv",
)

SMOKE_RANDOM_SEED = 20260808
SMOKE_NONZERO_POSITIVE_CASES = 1
SMOKE_ZERO_CONTROL_CASES = 1
SMOKE_EMPTY_NONZERO_MASK_CASES = 1

CLIP_MODEL_NAME = getattr(feature_similarity, "DEFAULT_CLIP_MODEL_NAME", "openai/clip-vit-base-patch32")
CLIP_MODEL_REVISION = getattr(feature_similarity, "DEFAULT_CLIP_MODEL_REVISION", "default")
DINOV2_MODEL_NAME = getattr(feature_similarity, "DEFAULT_DINOV2_MODEL_NAME", "dinov2_vits14")
DINOV2_MODEL_REVISION = getattr(feature_similarity, "DEFAULT_DINOV2_MODEL_REVISION", "torchhub-default")

FEATURE_INPUT_SIZE = int(globals().get(
    "FEATURE_INPUT_SIZE",
    getattr(feature_similarity, "DEFAULT_FEATURE_INPUT_SIZE", 224),
))
FEATURE_MASK_BINARY_THRESHOLD = int(globals().get(
    "FEATURE_MASK_BINARY_THRESHOLD",
    getattr(feature_similarity, "DEFAULT_MASK_BINARY_THRESHOLD", 127),
))
FEATURE_MASK_BBOX_MARGIN = int(globals().get(
    "FEATURE_MASK_BBOX_MARGIN",
    getattr(feature_similarity, "DEFAULT_MASK_BBOX_MARGIN", 8),
))

random.seed(SMOKE_RANDOM_SEED)
np.random.seed(SMOKE_RANDOM_SEED)

torch = importlib.import_module("torch")
torch.manual_seed(SMOKE_RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SMOKE_RANDOM_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

feature_runtime_dependency_df = feature_similarity.validate_feature_similarity_runtime_dependencies()
display(feature_runtime_dependency_df)

feature_runtime_dependencies_passed = bool_series(feature_runtime_dependency_df["passed"]).all()
if not feature_runtime_dependencies_passed:
    raise RuntimeError("Feature-similarity runtime dependencies are missing. Install/fix them before Batch 2.")

feature_device = feature_similarity.get_device(prefer_cuda=True)

model_load_started = perf_counter()
clip_model, clip_processor = feature_similarity.load_clip_model_and_processor(
    model_name=CLIP_MODEL_NAME,
    device=feature_device,
    revision=CLIP_MODEL_REVISION,
)
dinov2_model = feature_similarity.load_dinov2_model(
    model_name=DINOV2_MODEL_NAME,
    device=feature_device,
    trust_repo=True,
)
model_load_runtime_seconds = perf_counter() - model_load_started

print(f"Loaded CLIP: {CLIP_MODEL_NAME} ({CLIP_MODEL_REVISION})")
print(f"Loaded DINOv2: {DINOV2_MODEL_NAME} ({DINOV2_MODEL_REVISION})")
print(f"Device: {feature_device}")
print(f"Model load runtime: {model_load_runtime_seconds:.2f} seconds")

,component,module,version,required,installed,passed
0,torch,torch,2.5.1+cu121,True,True,True
1,torchvision,torchvision,0.20.1+cu121,True,True,True
2,transformers,transformers,4.48.3,True,True,True
3,jmespath,jmespath,1.1.0,True,True,True
4,safetensors,safetensors,0.8.0,True,True,True
5,Pillow,PIL,9.5.0,True,True,True


D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\transformers\utils\hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
C:\Users\rahul/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\rahul/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\rahul/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: 

Loaded CLIP: openai/clip-vit-base-patch32 (default)
Loaded DINOv2: dinov2_vits14 (torchhub-default)
Device: cuda
Model load runtime: 53.16 seconds


In [13]:
# Batch 2 / Cell 2 - Select deterministic smoke cases
feature_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

if not validation_file_passed(BATCH1_VALIDATION_PATH):
    raise RuntimeError("Batch 1 validation has not passed. Do not run Batch 2 yet.")

if "valid_for_feature_similarity" not in feature_input_cases_df.columns:
    raise ValueError("Batch 1 input cases are missing valid_for_feature_similarity.")

valid_feature_cases_df = feature_input_cases_df.loc[
    bool_series(feature_input_cases_df["valid_for_feature_similarity"])
].copy()

sort_columns = [
    column
    for column in [
        "dataset_name",
        "painting_id",
        "mask_type",
        "case_id",
        "candidate_id",
        "restoration_case_id",
    ]
    if column in valid_feature_cases_df.columns
]

valid_feature_cases_df = valid_feature_cases_df.sort_values(
    sort_columns,
    kind="stable",
).reset_index(drop=True)

nonzero_positive_smoke_df = (
    valid_feature_cases_df.loc[
        (~bool_series(valid_feature_cases_df["is_zero_control_for_feature_similarity"]))
        & bool_series(valid_feature_cases_df["mask_has_positive_pixels_for_feature_similarity"])
    ]
    .head(SMOKE_NONZERO_POSITIVE_CASES)
    .copy()
)
nonzero_positive_smoke_df["smoke_case_role"] = "nonzero_positive_mask"

zero_control_smoke_df = (
    valid_feature_cases_df.loc[
        bool_series(valid_feature_cases_df["is_zero_control_for_feature_similarity"])
    ]
    .head(SMOKE_ZERO_CONTROL_CASES)
    .copy()
)
zero_control_smoke_df["smoke_case_role"] = "zero_control"

empty_nonzero_mask_smoke_df = (
    valid_feature_cases_df.loc[
        (~bool_series(valid_feature_cases_df["is_zero_control_for_feature_similarity"]))
        & (~bool_series(valid_feature_cases_df["mask_has_positive_pixels_for_feature_similarity"]))
    ]
    .head(SMOKE_EMPTY_NONZERO_MASK_CASES)
    .copy()
)
empty_nonzero_mask_smoke_df["smoke_case_role"] = "empty_nonzero_mask"

feature_smoke_cases_df = (
    pd.concat(
        [
            nonzero_positive_smoke_df,
            zero_control_smoke_df,
            empty_nonzero_mask_smoke_df,
        ],
        ignore_index=True,
    )
    .sort_values(["smoke_case_role"] + sort_columns, kind="stable")
    .reset_index(drop=True)
)

if feature_smoke_cases_df.empty:
    raise RuntimeError("No smoke cases were selected.")

# Use absolute paths for helper execution, independent of notebook cwd.
feature_smoke_compute_df = feature_smoke_cases_df.copy()
for path_column in PATH_COLUMNS:
    feature_smoke_compute_df[path_column] = feature_smoke_compute_df[path_column].map(
        lambda value: str(resolve_project_path(value))
    )

smoke_expected_region_counts = feature_similarity.expected_feature_similarity_region_counts_from_metadata(
    feature_smoke_compute_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)
smoke_expected_feature_rows = feature_similarity.expected_feature_similarity_rows_from_metadata(
    feature_smoke_compute_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)

print(f"Smoke cases: {len(feature_smoke_cases_df)}")
print(f"Smoke expected region counts: {smoke_expected_region_counts}")
print(f"Smoke expected feature rows: {smoke_expected_feature_rows}")

display(
    feature_smoke_cases_df[
        [
            "smoke_case_role",
            "case_id",
            "candidate_id",
            "restoration_case_id",
            "mask_type",
            "mask_has_positive_pixels_for_feature_similarity",
            "expected_feature_regions",
        ]
    ]
)

Smoke cases: 3
Smoke expected region counts: {'full_image': 3, 'content_region': 3, 'mask_bbox_crop': 1}
Smoke expected feature rows: 7


,smoke_case_role,case_id,candidate_id,restoration_case_id,mask_type,mask_has_positive_pixels_for_feature_similarity,expected_feature_regions
0,empty_nonzero_mask,p026__water_stain__moderate,sd__syn__p026__water_stain__p00_generic__s2026...,sd__syn__p026__water_stain__p00_generic__s2026...,water_stain,False,full_image|content_region
1,nonzero_positive_mask,p001_loss_large,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,loss_large,True,full_image|content_region|mask_bbox_crop
2,zero_control,p001_zero_control,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,zero_control,False,full_image|content_region


In [14]:
# Batch 2 / Cell 3 - Run smoke feature-similarity metrics
smoke_metric_started = perf_counter()

feature_smoke_metrics_df = feature_similarity.compute_feature_similarity_for_restorations(
    restoration_metadata=feature_smoke_compute_df,
    clip_model=clip_model,
    clip_processor=clip_processor,
    dinov2_model=dinov2_model,
    device=feature_device,
    clip_model_name=CLIP_MODEL_NAME,
    clip_model_revision=CLIP_MODEL_REVISION,
    dinov2_model_name=DINOV2_MODEL_NAME,
    dinov2_model_revision=DINOV2_MODEL_REVISION,
    target_size=TARGET_SIZE,
    mask_bbox_margin=FEATURE_MASK_BBOX_MARGIN,
    feature_input_size=FEATURE_INPUT_SIZE,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
    progress_every=1,
)

smoke_metric_runtime_seconds = perf_counter() - smoke_metric_started

feature_smoke_metrics_df.to_csv(BATCH2_SMOKE_METRICS_PATH, index=False)

print(f"Smoke runtime: {smoke_metric_runtime_seconds:.2f} seconds")
print(f"Saved smoke metrics: {rel(BATCH2_SMOKE_METRICS_PATH)}")
print("Smoke status counts:")
print(feature_smoke_metrics_df["status"].value_counts(dropna=False).to_string())
print("Smoke region counts:")
print(feature_smoke_metrics_df["evaluation_region"].value_counts(dropna=False).to_string())

display(feature_smoke_metrics_df.head(10))

Starting feature-similarity metric computation
  Cases: 3
  Target size: (768, 768)
  Feature input size: 224
  CLIP model: openai/clip-vit-base-patch32 (default)
  DINOv2 model: dinov2_vits14 (torchhub-default)
  Device: cuda
Computing feature case 1/3 (p001_loss_large) | elapsed 0.01s
Computing feature case 2/3 (p001_zero_control) | elapsed 5.95s
Computing feature case 3/3 (p026__water_stain__moderate) | elapsed 6.45s
Feature-similarity metric computation complete
  Runtime: 6.95 seconds
  Output rows: 7
  Region counts:
evaluation_region
full_image        3
content_region    3
mask_bbox_crop    1
  Status counts:
status
ok    7
Smoke runtime: 6.96 seconds
Saved smoke metrics: outputs/25_stable_diffusion_feature_similarity/validation/stable_diffusion_feature_similarity_smoke.csv
Smoke status counts:
status
ok    7
Smoke region counts:
evaluation_region
full_image        3
content_region    3
mask_bbox_crop    1


,feature_row_id,feature_case_id,metric_case_id,restoration_case_id,source_case_id,source_case_id_original,case_id,dataset_name,metric_applicability,painting_id,...,device,python_version,numpy_version,pandas_version,pillow_version,torch_version,torchvision_version,transformers_version,status,issue
0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
1,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
2,sd24_0001__mask_bbox_crop__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
3,sd24_0005__full_image__feature_similarity,sd24_0005,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001_zero_control,p001_zero_control,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
4,sd24_0005__content_region__feature_similarity,sd24_0005,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001_zero_control,p001_zero_control,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
5,sd24_0826__full_image__feature_similarity,sd24_0826,sd24_0826,sd__syn__p026__water_stain__p00_generic__s2026...,p026__water_stain__moderate,p026__water_stain__moderate,p026__water_stain__moderate,synthetic_degradation,primary,p026,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
6,sd24_0826__content_region__feature_similarity,sd24_0826,sd24_0826,sd__syn__p026__water_stain__p00_generic__s2026...,p026__water_stain__moderate,p026__water_stain__moderate,p026__water_stain__moderate,synthetic_degradation,primary,p026,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,


In [15]:
# Batch 2 / Cell 4 - Validate smoke metrics and update stage manifest
smoke_helper_validation_df = feature_similarity.validate_feature_similarity_metrics(
    feature_smoke_metrics_df,
    expected_rows=smoke_expected_feature_rows,
    expected_region_counts=smoke_expected_region_counts,
)

smoke_role_counts = feature_smoke_cases_df["smoke_case_role"].value_counts().to_dict()
smoke_metric_region_counts = feature_smoke_metrics_df["evaluation_region"].value_counts().to_dict()
smoke_error_rows = int(feature_smoke_metrics_df["status"].eq("error").sum())

batch2_validation_rows = [
    validation_row(
        "batch1_validation_passed",
        validation_file_passed(BATCH1_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH1_VALIDATION_PATH),
        "Batch 1 validation did not pass.",
    ),
    validation_row(
        "smoke_case_count",
        len(feature_smoke_cases_df),
        SMOKE_NONZERO_POSITIVE_CASES + SMOKE_ZERO_CONTROL_CASES + SMOKE_EMPTY_NONZERO_MASK_CASES,
        len(feature_smoke_cases_df) == (
            SMOKE_NONZERO_POSITIVE_CASES + SMOKE_ZERO_CONTROL_CASES + SMOKE_EMPTY_NONZERO_MASK_CASES
        ),
        "Smoke case selection did not produce the expected number of cases.",
    ),
    validation_row(
        "smoke_case_roles_present",
        smoke_role_counts,
        {
            "nonzero_positive_mask": SMOKE_NONZERO_POSITIVE_CASES,
            "zero_control": SMOKE_ZERO_CONTROL_CASES,
            "empty_nonzero_mask": SMOKE_EMPTY_NONZERO_MASK_CASES,
        },
        smoke_role_counts == {
            "nonzero_positive_mask": SMOKE_NONZERO_POSITIVE_CASES,
            "zero_control": SMOKE_ZERO_CONTROL_CASES,
            "empty_nonzero_mask": SMOKE_EMPTY_NONZERO_MASK_CASES,
        },
        "Smoke case roles are incomplete or incorrectly counted.",
    ),
    validation_row(
        "smoke_expected_feature_rows",
        smoke_expected_feature_rows,
        7,
        smoke_expected_feature_rows == 7,
        "Smoke expected feature-row count should be 7.",
    ),
    validation_row(
        "smoke_expected_region_counts",
        smoke_expected_region_counts,
        {"full_image": 3, "content_region": 3, "mask_bbox_crop": 1},
        smoke_expected_region_counts == {"full_image": 3, "content_region": 3, "mask_bbox_crop": 1},
        "Smoke expected region counts are wrong.",
    ),
    validation_row(
        "smoke_metrics_row_count",
        len(feature_smoke_metrics_df),
        smoke_expected_feature_rows,
        len(feature_smoke_metrics_df) == smoke_expected_feature_rows,
        "Smoke metrics row count does not match expected feature rows.",
    ),
    validation_row(
        "smoke_metrics_region_counts",
        smoke_metric_region_counts,
        smoke_expected_region_counts,
        smoke_metric_region_counts == smoke_expected_region_counts,
        "Smoke metric region counts do not match expected region counts.",
    ),
    validation_row(
        "smoke_no_error_rows",
        smoke_error_rows,
        0,
        smoke_error_rows == 0,
        "Smoke feature-similarity computation produced error rows.",
    ),
    validation_row(
        "smoke_metrics_written",
        rel(BATCH2_SMOKE_METRICS_PATH),
        "file exists",
        BATCH2_SMOKE_METRICS_PATH.is_file(),
        "Smoke metrics CSV was not written.",
    ),
]

for _, helper_row in smoke_helper_validation_df.iterrows():
    passed = bool(helper_row["passed"])
    batch2_validation_rows.append(
        validation_row(
            f"helper_{helper_row['check']}",
            helper_row["detail"],
            "passed",
            passed,
            "" if passed else helper_row["detail"],
        )
    )

batch2_validation_df = pd.DataFrame(batch2_validation_rows)
batch2_validation_df.to_csv(BATCH2_VALIDATION_PATH, index=False)

batch2_passed = bool_series(batch2_validation_df["passed"]).all()

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch2_model_load_and_smoke_feature_similarity"
stage_manifest["stage_status"] = "passed" if batch2_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch2"] = {
    "status": "passed" if batch2_passed else "failed",
    "smoke_metrics": rel(BATCH2_SMOKE_METRICS_PATH),
    "validation": rel(BATCH2_VALIDATION_PATH),
    "smoke_cases": int(len(feature_smoke_cases_df)),
    "smoke_case_roles": smoke_role_counts,
    "expected_region_counts": smoke_expected_region_counts,
    "expected_feature_rows": int(smoke_expected_feature_rows),
    "actual_region_counts": smoke_metric_region_counts,
    "actual_feature_rows": int(len(feature_smoke_metrics_df)),
    "error_rows": smoke_error_rows,
    "clip_model_name": CLIP_MODEL_NAME,
    "clip_model_revision": CLIP_MODEL_REVISION,
    "dinov2_model_name": DINOV2_MODEL_NAME,
    "dinov2_model_revision": DINOV2_MODEL_REVISION,
    "device": str(feature_device),
    "feature_input_size": FEATURE_INPUT_SIZE,
    "mask_binary_threshold": FEATURE_MASK_BINARY_THRESHOLD,
    "mask_bbox_margin": FEATURE_MASK_BBOX_MARGIN,
    "model_load_runtime_seconds": float(model_load_runtime_seconds),
    "smoke_metric_runtime_seconds": float(smoke_metric_runtime_seconds),
    "checks_passed": int(bool_series(batch2_validation_df["passed"]).sum()),
    "checks_total": int(len(batch2_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved smoke validation: {rel(BATCH2_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 2 checks passed: {int(bool_series(batch2_validation_df['passed']).sum())} / {len(batch2_validation_df)}")

display(batch2_validation_df)

if not batch2_passed:
    display(batch2_validation_df.loc[~bool_series(batch2_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 2 smoke validation failed. Fix model/runtime/smoke issues before Batch 3.")

print("Batch 2 passed. Continue to Batch 3.")

Saved smoke validation: outputs/25_stable_diffusion_feature_similarity/validation/batch2_smoke_validation.csv
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 2 checks passed: 19 / 19


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch1_validation_passed,True,True,True,,2026-08-08T17:44:18+00:00
1,smoke_case_count,3,3,True,,2026-08-08T17:44:18+00:00
2,smoke_case_roles_present,"{'empty_nonzero_mask': 1, 'nonzero_positive_ma...","{'nonzero_positive_mask': 1, 'zero_control': 1...",True,,2026-08-08T17:44:18+00:00
3,smoke_expected_feature_rows,7,7,True,,2026-08-08T17:44:18+00:00
4,smoke_expected_region_counts,"{'full_image': 3, 'content_region': 3, 'mask_b...","{'full_image': 3, 'content_region': 3, 'mask_b...",True,,2026-08-08T17:44:18+00:00
5,smoke_metrics_row_count,7,7,True,,2026-08-08T17:44:18+00:00
6,smoke_metrics_region_counts,"{'full_image': 3, 'content_region': 3, 'mask_b...","{'full_image': 3, 'content_region': 3, 'mask_b...",True,,2026-08-08T17:44:18+00:00
7,smoke_no_error_rows,0,0,True,,2026-08-08T17:44:18+00:00
8,smoke_metrics_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T17:44:18+00:00
9,helper_required_columns,All required columns present.,passed,True,,2026-08-08T17:44:18+00:00


Batch 2 passed. Continue to Batch 3.


In [16]:
# Batch 3 / Cell 1 - Paths and full input preparation
BATCH3_METRICS_PATH = globals().get(
    "BATCH3_METRICS_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_metrics.csv",
)
BATCH3_EMBEDDINGS_NPZ_PATH = globals().get(
    "BATCH3_EMBEDDINGS_NPZ_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_embeddings.npz",
)
BATCH3_EMBEDDING_MANIFEST_PATH = globals().get(
    "BATCH3_EMBEDDING_MANIFEST_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_embedding_manifest.csv",
)
BATCH3_VALIDATION_PATH = globals().get(
    "BATCH3_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_full_validation.csv",
)

BATCH3_PROGRESS_EVERY = int(globals().get("BATCH3_PROGRESS_EVERY", 25))
BATCH3_EMBEDDING_DTYPE = str(globals().get("BATCH3_EMBEDDING_DTYPE", "float32"))
EXPECTED_EMBEDDING_VECTORS_PER_FEATURE_ROW = 6

required_batch3_runtime_objects = [
    "clip_model",
    "clip_processor",
    "dinov2_model",
    "feature_device",
]
missing_batch3_runtime_objects = [
    name for name in required_batch3_runtime_objects
    if name not in globals()
]
if missing_batch3_runtime_objects:
    raise RuntimeError(
        "Batch 3 needs Batch 2 model objects still loaded. Missing: "
        f"{missing_batch3_runtime_objects}. Rerun Batch 2 / Cell 1."
    )

if not validation_file_passed(BATCH1_VALIDATION_PATH):
    raise RuntimeError("Batch 1 validation has not passed. Do not run Batch 3.")

if not validation_file_passed(BATCH2_VALIDATION_PATH):
    raise RuntimeError("Batch 2 validation has not passed. Do not run Batch 3.")

feature_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

valid_full_feature_cases_df = feature_input_cases_df.loc[
    bool_series(feature_input_cases_df["valid_for_feature_similarity"])
].copy()

if len(valid_full_feature_cases_df) != EXPECTED_CANDIDATE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_CANDIDATE_ROWS:,} valid feature cases, "
        f"found {len(valid_full_feature_cases_df):,}."
    )

full_feature_compute_df = valid_full_feature_cases_df.copy()
for path_column in PATH_COLUMNS:
    full_feature_compute_df[path_column] = full_feature_compute_df[path_column].map(
        lambda value: str(resolve_project_path(value))
    )

full_expected_region_counts = feature_similarity.expected_feature_similarity_region_counts_from_metadata(
    full_feature_compute_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)
full_expected_feature_rows = feature_similarity.expected_feature_similarity_rows_from_metadata(
    full_feature_compute_df,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
)
full_expected_embedding_rows = (
    int(full_expected_feature_rows) * EXPECTED_EMBEDDING_VECTORS_PER_FEATURE_ROW
)

print(f"Full feature cases: {len(full_feature_compute_df):,}")
print(f"Expected feature rows: {full_expected_feature_rows:,}")
print(f"Expected region counts: {full_expected_region_counts}")
print(f"Expected embedding rows/vectors: {full_expected_embedding_rows:,}")

display(
    pd.DataFrame(
        [
            {
                "artifact": "feature_metrics_csv",
                "path": rel(BATCH3_METRICS_PATH),
            },
            {
                "artifact": "embedding_manifest_csv",
                "path": rel(BATCH3_EMBEDDING_MANIFEST_PATH),
            },
            {
                "artifact": "embeddings_npz",
                "path": rel(BATCH3_EMBEDDINGS_NPZ_PATH),
            },
            {
                "artifact": "full_validation_csv",
                "path": rel(BATCH3_VALIDATION_PATH),
            },
        ]
    )
)

Full feature cases: 945
Expected feature rows: 2,724
Expected region counts: {'full_image': 945, 'content_region': 945, 'mask_bbox_crop': 834}
Expected embedding rows/vectors: 16,344


,artifact,path
0,feature_metrics_csv,outputs/25_stable_diffusion_feature_similarity...
1,embedding_manifest_csv,outputs/25_stable_diffusion_feature_similarity...
2,embeddings_npz,outputs/25_stable_diffusion_feature_similarity...
3,full_validation_csv,outputs/25_stable_diffusion_feature_similarity...


In [17]:
# Batch 3 / Cell 2 - Full CLIP/DINOv2 feature metrics and retained embeddings
batch3_started_at_utc = utc_now_iso()
batch3_metric_started = perf_counter()

(
    stable_diffusion_feature_metrics_df,
    stable_diffusion_embedding_manifest_df,
    stable_diffusion_embedding_arrays,
) = feature_similarity.compute_feature_similarity_for_restorations_with_embeddings(
    restoration_metadata=full_feature_compute_df,
    clip_model=clip_model,
    clip_processor=clip_processor,
    dinov2_model=dinov2_model,
    device=feature_device,
    clip_model_name=CLIP_MODEL_NAME,
    clip_model_revision=CLIP_MODEL_REVISION,
    dinov2_model_name=DINOV2_MODEL_NAME,
    dinov2_model_revision=DINOV2_MODEL_REVISION,
    target_size=TARGET_SIZE,
    mask_bbox_margin=FEATURE_MASK_BBOX_MARGIN,
    feature_input_size=FEATURE_INPUT_SIZE,
    mask_binary_threshold=FEATURE_MASK_BINARY_THRESHOLD,
    progress_every=BATCH3_PROGRESS_EVERY,
    embedding_dtype=BATCH3_EMBEDDING_DTYPE,
)

batch3_metric_runtime_seconds = perf_counter() - batch3_metric_started

stable_diffusion_feature_metrics_df.to_csv(BATCH3_METRICS_PATH, index=False)
stable_diffusion_embedding_manifest_df.to_csv(BATCH3_EMBEDDING_MANIFEST_PATH, index=False)

feature_similarity.save_feature_embedding_bundle(
    stable_diffusion_embedding_arrays,
    output_path=BATCH3_EMBEDDINGS_NPZ_PATH,
    dtype=BATCH3_EMBEDDING_DTYPE,
    compressed=True,
)

print(f"Full feature runtime: {batch3_metric_runtime_seconds:.2f} seconds")
print(f"Saved metrics: {rel(BATCH3_METRICS_PATH)}")
print(f"Saved embedding manifest: {rel(BATCH3_EMBEDDING_MANIFEST_PATH)}")
print(f"Saved embeddings NPZ: {rel(BATCH3_EMBEDDINGS_NPZ_PATH)}")

print("Metric status counts:")
print(stable_diffusion_feature_metrics_df["status"].value_counts(dropna=False).to_string())

print("Metric region counts:")
print(stable_diffusion_feature_metrics_df["evaluation_region"].value_counts(dropna=False).to_string())

print("Embedding manifest model/image-role counts:")
display(
    stable_diffusion_embedding_manifest_df
    .groupby(["model_family", "image_role"])
    .size()
    .reset_index(name="rows")
)

display(stable_diffusion_feature_metrics_df.head(10))
display(stable_diffusion_embedding_manifest_df.head(10))

Starting feature-similarity metric computation
  Cases: 945
  Target size: (768, 768)
  Feature input size: 224
  CLIP model: openai/clip-vit-base-patch32 (default)
  DINOv2 model: dinov2_vits14 (torchhub-default)
  Device: cuda
Computing feature case 1/945 (p001_loss_large) | elapsed 0.02s
Computing feature case 25/945 (p004_loss_small) | elapsed 15.67s
Computing feature case 50/945 (p006_zero_control) | elapsed 31.49s
Computing feature case 75/945 (p008_scratch_thin) | elapsed 48.00s
Computing feature case 100/945 (p012_zero_control) | elapsed 63.35s
Computing feature case 125/945 (p014_scratch_thin) | elapsed 78.72s
Computing feature case 150/945 (p018_loss_large) | elapsed 93.45s
Computing feature case 175/945 (p019_zero_control) | elapsed 108.64s
Computing feature case 200/945 (p024_loss_large) | elapsed 124.55s
Computing feature case 225/945 (p025_zero_control) | elapsed 140.52s
Computing feature case 250/945 (p028_mixed_damage) | elapsed 156.53s
Computing feature case 275/945 (p

,model_family,image_role,rows
0,clip,clean,2724
1,clip,damaged,2724
2,clip,restored,2724
3,dinov2,clean,2724
4,dinov2,damaged,2724
5,dinov2,restored,2724


,feature_row_id,feature_case_id,metric_case_id,restoration_case_id,source_case_id,source_case_id_original,case_id,dataset_name,metric_applicability,painting_id,...,device,python_version,numpy_version,pandas_version,pillow_version,torch_version,torchvision_version,transformers_version,status,issue
0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
1,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
2,sd24_0001__mask_bbox_crop__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
3,sd24_0002__full_image__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
4,sd24_0002__content_region__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
5,sd24_0002__mask_bbox_crop__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
6,sd24_0003__full_image__feature_similarity,sd24_0003,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
7,sd24_0003__content_region__feature_similarity,sd24_0003,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
8,sd24_0003__mask_bbox_crop__feature_similarity,sd24_0003,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,
9,sd24_0004__full_image__feature_similarity,sd24_0004,sd24_0004,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,p001_scratch_thin,p001_scratch_thin,canonical,primary,p001,...,cuda,3.12.6,1.26.4,2.3.3,9.5.0,2.5.1+cu121,0.20.1+cu121,4.48.3,ok,


,embedding_manifest_schema_version,feature_row_id,feature_case_id,metric_case_id,restoration_case_id,source_case_id,source_case_id_original,case_id,dataset_name,painting_id,...,npz_key,vector_dim,dtype,feature_input_size,clip_model_name,clip_model_revision,dinov2_model_name,dinov2_model_revision,device,status
0,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__cli...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
1,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__cli...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
2,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__cli...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
3,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__din...,384,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
4,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__din...,384,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
5,1.0.0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__full_image__feature_similarity__din...,384,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
6,1.0.0,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__content_region__feature_similarity_...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
7,1.0.0,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__content_region__feature_similarity_...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
8,1.0.0,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__content_region__feature_similarity_...,512,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok
9,1.0.0,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,p001,...,sd24_0001__content_region__feature_similarity_...,384,float32,224,openai/clip-vit-base-patch32,default,dinov2_vits14,torchhub-default,cuda,ok


In [18]:
# Batch 3 / Cell 3 - Full validation and stage manifest update
metrics_validation_df = feature_similarity.validate_feature_similarity_metrics(
    stable_diffusion_feature_metrics_df,
    expected_rows=full_expected_feature_rows,
    expected_region_counts=full_expected_region_counts,
)

embedding_manifest_validation_df = feature_similarity.validate_feature_embedding_manifest(
    stable_diffusion_embedding_manifest_df,
    embedding_arrays=stable_diffusion_embedding_arrays,
    expected_feature_rows=full_expected_feature_rows,
    expected_vectors_per_feature_row=EXPECTED_EMBEDDING_VECTORS_PER_FEATURE_ROW,
)

loaded_npz = np.load(BATCH3_EMBEDDINGS_NPZ_PATH)
npz_keys = list(loaded_npz.files)
loaded_npz.close()

metric_error_rows = int(stable_diffusion_feature_metrics_df["status"].eq("error").sum())
actual_region_counts = stable_diffusion_feature_metrics_df["evaluation_region"].value_counts().to_dict()
actual_embedding_rows = int(len(stable_diffusion_embedding_manifest_df))
actual_embedding_arrays = int(len(stable_diffusion_embedding_arrays))
actual_npz_arrays = int(len(npz_keys))

batch3_validation_rows = [
    validation_row(
        "batch1_validation_passed",
        validation_file_passed(BATCH1_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH1_VALIDATION_PATH),
        "Batch 1 validation did not pass.",
    ),
    validation_row(
        "batch2_validation_passed",
        validation_file_passed(BATCH2_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH2_VALIDATION_PATH),
        "Batch 2 validation did not pass.",
    ),
    validation_row(
        "full_feature_case_count",
        len(full_feature_compute_df),
        EXPECTED_CANDIDATE_ROWS,
        len(full_feature_compute_df) == EXPECTED_CANDIDATE_ROWS,
        "Full feature input case count does not match the Notebook 25 contract.",
    ),
    validation_row(
        "full_expected_feature_rows",
        full_expected_feature_rows,
        EXPECTED_FEATURE_ROWS,
        full_expected_feature_rows == EXPECTED_FEATURE_ROWS,
        "Expected full feature rows do not match the 2,724-row contract.",
    ),
    validation_row(
        "full_expected_region_counts",
        full_expected_region_counts,
        EXPECTED_REGION_COUNTS,
        full_expected_region_counts == EXPECTED_REGION_COUNTS,
        "Expected full region counts do not match the Notebook 25 contract.",
    ),
    validation_row(
        "feature_metrics_row_count",
        len(stable_diffusion_feature_metrics_df),
        full_expected_feature_rows,
        len(stable_diffusion_feature_metrics_df) == full_expected_feature_rows,
        "Feature metrics row count does not match expected rows.",
    ),
    validation_row(
        "feature_metrics_region_counts",
        actual_region_counts,
        full_expected_region_counts,
        actual_region_counts == full_expected_region_counts,
        "Feature metric region counts do not match expected counts.",
    ),
    validation_row(
        "feature_metrics_no_error_rows",
        metric_error_rows,
        0,
        metric_error_rows == 0,
        "Feature metric computation produced error rows.",
    ),
    validation_row(
        "embedding_manifest_row_count",
        actual_embedding_rows,
        full_expected_embedding_rows,
        actual_embedding_rows == full_expected_embedding_rows,
        "Embedding manifest row count does not match expected vectors.",
    ),
    validation_row(
        "embedding_array_count",
        actual_embedding_arrays,
        full_expected_embedding_rows,
        actual_embedding_arrays == full_expected_embedding_rows,
        "Retained embedding array count does not match expected vectors.",
    ),
    validation_row(
        "embedding_npz_array_count",
        actual_npz_arrays,
        full_expected_embedding_rows,
        actual_npz_arrays == full_expected_embedding_rows,
        "Saved NPZ array count does not match expected vectors.",
    ),
    validation_row(
        "feature_metrics_written",
        rel(BATCH3_METRICS_PATH),
        "file exists",
        BATCH3_METRICS_PATH.is_file(),
        "Feature metrics CSV was not written.",
    ),
    validation_row(
        "embedding_manifest_written",
        rel(BATCH3_EMBEDDING_MANIFEST_PATH),
        "file exists",
        BATCH3_EMBEDDING_MANIFEST_PATH.is_file(),
        "Embedding manifest CSV was not written.",
    ),
    validation_row(
        "embeddings_npz_written",
        rel(BATCH3_EMBEDDINGS_NPZ_PATH),
        "file exists",
        BATCH3_EMBEDDINGS_NPZ_PATH.is_file(),
        "Embeddings NPZ was not written.",
    ),
]

for _, helper_row in metrics_validation_df.iterrows():
    passed = bool(helper_row["passed"])
    batch3_validation_rows.append(
        validation_row(
            f"helper_metrics_{helper_row['check']}",
            helper_row["detail"],
            "passed",
            passed,
            "" if passed else helper_row["detail"],
        )
    )

for _, helper_row in embedding_manifest_validation_df.iterrows():
    passed = bool(helper_row["passed"])
    batch3_validation_rows.append(
        validation_row(
            f"helper_embeddings_{helper_row['check']}",
            helper_row["detail"],
            "passed",
            passed,
            "" if passed else helper_row["detail"],
        )
    )

batch3_validation_df = pd.DataFrame(batch3_validation_rows)
batch3_validation_df.to_csv(BATCH3_VALIDATION_PATH, index=False)

batch3_passed = bool_series(batch3_validation_df["passed"]).all()

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch3_full_feature_similarity_with_embeddings"
stage_manifest["stage_status"] = "passed" if batch3_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch3"] = {
    "status": "passed" if batch3_passed else "failed",
    "started_at_utc": batch3_started_at_utc,
    "finished_at_utc": utc_now_iso(),
    "metrics": rel(BATCH3_METRICS_PATH),
    "embedding_manifest": rel(BATCH3_EMBEDDING_MANIFEST_PATH),
    "embeddings_npz": rel(BATCH3_EMBEDDINGS_NPZ_PATH),
    "validation": rel(BATCH3_VALIDATION_PATH),
    "input_cases": int(len(full_feature_compute_df)),
    "expected_region_counts": full_expected_region_counts,
    "actual_region_counts": actual_region_counts,
    "expected_feature_rows": int(full_expected_feature_rows),
    "actual_feature_rows": int(len(stable_diffusion_feature_metrics_df)),
    "expected_embedding_rows": int(full_expected_embedding_rows),
    "actual_embedding_manifest_rows": actual_embedding_rows,
    "actual_embedding_arrays": actual_embedding_arrays,
    "actual_npz_arrays": actual_npz_arrays,
    "error_rows": metric_error_rows,
    "clip_model_name": CLIP_MODEL_NAME,
    "clip_model_revision": CLIP_MODEL_REVISION,
    "dinov2_model_name": DINOV2_MODEL_NAME,
    "dinov2_model_revision": DINOV2_MODEL_REVISION,
    "device": str(feature_device),
    "feature_input_size": FEATURE_INPUT_SIZE,
    "embedding_dtype": BATCH3_EMBEDDING_DTYPE,
    "mask_binary_threshold": FEATURE_MASK_BINARY_THRESHOLD,
    "mask_bbox_margin": FEATURE_MASK_BBOX_MARGIN,
    "runtime_seconds": float(batch3_metric_runtime_seconds),
    "checks_passed": int(bool_series(batch3_validation_df["passed"]).sum()),
    "checks_total": int(len(batch3_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved full validation: {rel(BATCH3_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 3 checks passed: {int(bool_series(batch3_validation_df['passed']).sum())} / {len(batch3_validation_df)}")

display(batch3_validation_df)

if not batch3_passed:
    display(batch3_validation_df.loc[~bool_series(batch3_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 3 full validation failed. Fix feature/embedding issues before Batch 4.")

print("Batch 3 passed. Continue to Batch 4.")

Saved full validation: outputs/25_stable_diffusion_feature_similarity/validation/stable_diffusion_feature_similarity_metrics_validation.csv
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 3 checks passed: 31 / 31


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch1_validation_passed,True,True,True,,2026-08-08T18:01:19+00:00
1,batch2_validation_passed,True,True,True,,2026-08-08T18:01:19+00:00
2,full_feature_case_count,945,945,True,,2026-08-08T18:01:19+00:00
3,full_expected_feature_rows,2724,2724,True,,2026-08-08T18:01:19+00:00
4,full_expected_region_counts,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,,2026-08-08T18:01:19+00:00
5,feature_metrics_row_count,2724,2724,True,,2026-08-08T18:01:19+00:00
6,feature_metrics_region_counts,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,,2026-08-08T18:01:19+00:00
7,feature_metrics_no_error_rows,0,0,True,,2026-08-08T18:01:19+00:00
8,embedding_manifest_row_count,16344,16344,True,,2026-08-08T18:01:19+00:00
9,embedding_array_count,16344,16344,True,,2026-08-08T18:01:19+00:00


Batch 3 passed. Continue to Batch 4.


In [19]:
# Batch 4 / Cell 1 - Paths, load Batch 3 metrics, and enrich metadata
BATCH4_SUMMARY_PATH = globals().get(
    "BATCH4_SUMMARY_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_summary.csv",
)
BATCH4_SELECTED_CASES_PATH = globals().get(
    "BATCH4_SELECTED_CASES_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_selected_cases.csv",
)
BATCH4_VALIDATION_PATH = globals().get(
    "BATCH4_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv",
)

BATCH4_SELECTION_TOP_N = int(globals().get("BATCH4_SELECTION_TOP_N", 12))

if not validation_file_passed(BATCH3_VALIDATION_PATH):
    raise RuntimeError("Batch 3 validation has not passed. Do not run Batch 4 yet.")

if not BATCH3_METRICS_PATH.is_file():
    raise FileNotFoundError(f"Missing Batch 3 metrics CSV: {BATCH3_METRICS_PATH}")

if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(f"Missing Batch 1 input cases CSV: {BATCH1_INPUT_CASES_PATH}")

feature_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
feature_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

required_batch4_metric_columns = [
    "feature_row_id",
    "feature_case_id",
    "restoration_case_id",
    "case_id",
    "dataset_name",
    "painting_id",
    "model_name",
    "mask_type",
    "evaluation_region",
    "clip_damaged_similarity",
    "clip_restored_similarity",
    "clip_similarity_improvement",
    "dinov2_damaged_similarity",
    "dinov2_restored_similarity",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
    "status",
]
missing_batch4_metric_columns = [
    column for column in required_batch4_metric_columns
    if column not in feature_metrics_df.columns
]
if missing_batch4_metric_columns:
    raise ValueError(f"Batch 4 metrics input missing columns: {missing_batch4_metric_columns}")

metadata_enrichment_columns = [
    "restoration_case_id",
    "candidate_id",
    "case_id",
    "source_case_id",
    "source_case_id_original",
    "dataset_name",
    "painting_id",
    "category",
    "title",
    "artist",
    "style",
    "style_or_period",
    "mask_id",
    "mask_type",
    "metric_mask_id",
    "metric_mask_type",
    "model_name",
    "prompt_variant",
    "prompt_strategy",
    "guidance_scale",
    "num_inference_steps",
    "seed",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "is_zero_control_for_feature_similarity",
    "mask_has_positive_pixels_for_feature_similarity",
    "mask_positive_pixel_count_for_feature_similarity",
    "expected_feature_regions",
]
metadata_enrichment_columns = [
    column for column in metadata_enrichment_columns
    if column in feature_input_cases_df.columns
]

metadata_lookup_df = (
    feature_input_cases_df[metadata_enrichment_columns]
    .drop_duplicates(subset=["restoration_case_id"])
    .copy()
)

metadata_lookup_df = metadata_lookup_df.rename(
    columns={
        column: f"{column}_input"
        for column in metadata_lookup_df.columns
        if column != "restoration_case_id"
    }
)

batch4_metrics_df = feature_metrics_df.merge(
    metadata_lookup_df,
    on="restoration_case_id",
    how="left",
    validate="many_to_one",
)

# Prefer metric columns as source of truth, but fill useful missing context from Batch 1.
for base_column in [
    "candidate_id",
    "category",
    "title",
    "artist",
    "style",
    "style_or_period",
    "prompt_variant",
    "prompt_strategy",
    "guidance_scale",
    "num_inference_steps",
    "seed",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "is_zero_control_for_feature_similarity",
    "mask_has_positive_pixels_for_feature_similarity",
    "mask_positive_pixel_count_for_feature_similarity",
    "expected_feature_regions",
]:
    input_column = f"{base_column}_input"
    if input_column in batch4_metrics_df.columns:
        if base_column not in batch4_metrics_df.columns:
            batch4_metrics_df[base_column] = batch4_metrics_df[input_column]
        else:
            batch4_metrics_df[base_column] = batch4_metrics_df[base_column].where(
                batch4_metrics_df[base_column].notna(),
                batch4_metrics_df[input_column],
            )

if "style" not in batch4_metrics_df.columns and "style_or_period" in batch4_metrics_df.columns:
    batch4_metrics_df["style"] = batch4_metrics_df["style_or_period"]

for column in [
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
    "clip_damaged_similarity",
    "clip_restored_similarity",
    "dinov2_damaged_similarity",
    "dinov2_restored_similarity",
]:
    batch4_metrics_df[column] = pd.to_numeric(batch4_metrics_df[column], errors="coerce")

ok_feature_metrics_df = batch4_metrics_df.loc[
    batch4_metrics_df["status"].astype(str).eq("ok")
].copy()

print(f"Loaded Batch 3 metrics: {len(feature_metrics_df):,}")
print(f"OK feature metrics: {len(ok_feature_metrics_df):,}")
print("Region counts:")
print(ok_feature_metrics_df["evaluation_region"].value_counts(dropna=False).to_string())

display(ok_feature_metrics_df.head(10))

Loaded Batch 3 metrics: 2,724
OK feature metrics: 2,724
Region counts:
evaluation_region
full_image        945
content_region    945
mask_bbox_crop    834


,feature_row_id,feature_case_id,metric_case_id,restoration_case_id,source_case_id,source_case_id_original,case_id,dataset_name,metric_applicability,painting_id,...,guidance_scale,num_inference_steps,clean_path,damaged_path,restored_path,mask_path,is_zero_control_for_feature_similarity,mask_has_positive_pixels_for_feature_similarity,mask_positive_pixel_count_for_feature_similarity,expected_feature_regions
0,sd24_0001__full_image__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png,False,True,69018,full_image|content_region|mask_bbox_crop
1,sd24_0001__content_region__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png,False,True,69018,full_image|content_region|mask_bbox_crop
2,sd24_0001__mask_bbox_crop__feature_similarity,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,p001_loss_large,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png,False,True,69018,full_image|content_region|mask_bbox_crop
3,sd24_0002__full_image__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png,False,True,18239,full_image|content_region|mask_bbox_crop
4,sd24_0002__content_region__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png,False,True,18239,full_image|content_region|mask_bbox_crop
5,sd24_0002__mask_bbox_crop__feature_similarity,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,p001_loss_small,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png,False,True,18239,full_image|content_region|mask_bbox_crop
6,sd24_0003__full_image__feature_similarity,sd24_0003,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_mixed_damage_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_mixed_damage_mask.png,False,True,45733,full_image|content_region|mask_bbox_crop
7,sd24_0003__content_region__feature_similarity,sd24_0003,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001_mixed_damage,p001_mixed_damage,canonical,primary,p001,...,7.5,30,data/processed/clean/p001_clean.png,data/processed/masked/p001_mixed_damage_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_mixed_damage_mask.png,False,True,45733,full_image|content_region|mask_bbox_crop
8,sd24_0003__mask_bbox_crop__feature_similarity,sd24_0003,sd24_0003,

In [20]:
# Batch 4 / Cell 2 - Add ranking and disagreement diagnostics
analysis_metrics_df = ok_feature_metrics_df.copy()

analysis_metrics_df["clip_improved"] = analysis_metrics_df["clip_similarity_improvement"].gt(0)
analysis_metrics_df["dinov2_improved"] = analysis_metrics_df["dinov2_similarity_improvement"].gt(0)
analysis_metrics_df["both_feature_models_improved"] = (
    analysis_metrics_df["clip_improved"] & analysis_metrics_df["dinov2_improved"]
)
analysis_metrics_df["both_feature_models_worsened_or_flat"] = (
    ~analysis_metrics_df["clip_improved"] & ~analysis_metrics_df["dinov2_improved"]
)
analysis_metrics_df["feature_model_direction_disagreement"] = (
    analysis_metrics_df["clip_improved"] != analysis_metrics_df["dinov2_improved"]
)
analysis_metrics_df["clip_minus_dinov2_improvement"] = (
    analysis_metrics_df["clip_similarity_improvement"]
    - analysis_metrics_df["dinov2_similarity_improvement"]
)
analysis_metrics_df["absolute_feature_model_improvement_gap"] = (
    analysis_metrics_df["clip_minus_dinov2_improvement"].abs()
)

analysis_metrics_df["feature_direction_label"] = np.select(
    [
        analysis_metrics_df["both_feature_models_improved"],
        analysis_metrics_df["clip_improved"] & ~analysis_metrics_df["dinov2_improved"],
        ~analysis_metrics_df["clip_improved"] & analysis_metrics_df["dinov2_improved"],
    ],
    [
        "both_improved",
        "clip_improved_dinov2_not",
        "dinov2_improved_clip_not",
    ],
    default="neither_improved",
)

analysis_metrics_df["mean_similarity_rank_desc_by_region"] = (
    analysis_metrics_df
    .groupby("evaluation_region")["mean_similarity_improvement"]
    .rank(method="first", ascending=False)
    .astype(int)
)
analysis_metrics_df["mean_similarity_rank_asc_by_region"] = (
    analysis_metrics_df
    .groupby("evaluation_region")["mean_similarity_improvement"]
    .rank(method="first", ascending=True)
    .astype(int)
)
analysis_metrics_df["disagreement_gap_rank_desc_by_region"] = (
    analysis_metrics_df
    .groupby("evaluation_region")["absolute_feature_model_improvement_gap"]
    .rank(method="first", ascending=False)
    .astype(int)
)

print("Feature direction labels:")
print(analysis_metrics_df["feature_direction_label"].value_counts(dropna=False).to_string())

display(
    analysis_metrics_df[
        [
            "feature_case_id",
            "candidate_id",
            "evaluation_region",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
            "mean_similarity_improvement",
            "feature_direction_label",
            "absolute_feature_model_improvement_gap",
        ]
    ].head(12)
)

Feature direction labels:
feature_direction_label
both_improved               1490
neither_improved            1033
clip_improved_dinov2_not     201


,feature_case_id,candidate_id,evaluation_region,clip_similarity_improvement,dinov2_similarity_improvement,mean_similarity_improvement,feature_direction_label,absolute_feature_model_improvement_gap
0,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,full_image,0.143974,0.076536,0.110255,both_improved,0.067438
1,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,content_region,0.135581,0.089237,0.112409,both_improved,0.046344
2,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,mask_bbox_crop,0.233165,0.174986,0.204075,both_improved,0.058179
3,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,full_image,0.116053,0.082399,0.099226,both_improved,0.033655
4,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,content_region,0.107553,0.078576,0.093064,both_improved,0.028977
5,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,mask_bbox_crop,0.119281,0.057297,0.088289,both_improved,0.061984
6,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,full_image,0.237880,0.196223,0.217052,both_improved,0.041658
7,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,content_region,0.188747,0.217740,0.203244,both_improved,0.028993
8,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,mask_bbox_crop,0.237880,0.196223,0.217052,both_improved,0.041658
9,sd24_0004,sd__can__p001__scratch_thin__p00_generic__s202...,full_image,0.104852,0.048871,0.076862,both_improved,0.055981


In [21]:
# Batch 4 / Cell 3 - Build multi-scope summary CSV
def safe_rate(numerator: int | float, denominator: int | float) -> float:
    denominator = float(denominator)
    if denominator == 0:
        return np.nan
    return round(float(numerator) / denominator, 6)


def summarize_feature_group(group_df: pd.DataFrame, summary_scope: str, group_columns: list[str]) -> dict:
    case_count = int(group_df["feature_case_id"].nunique())
    row_count = int(len(group_df))

    record = {
        "summary_scope": summary_scope,
        "summary_key": "all_cases" if not group_columns else " | ".join(
            str(group_df[column].iloc[0]) for column in group_columns
        ),
        "group_columns": "|".join(group_columns),
        "row_count": row_count,
        "feature_case_count": case_count,
        "unique_paintings": int(group_df["painting_id"].nunique()) if "painting_id" in group_df.columns else np.nan,
        "mean_clip_damaged_similarity": group_df["clip_damaged_similarity"].mean(),
        "mean_clip_restored_similarity": group_df["clip_restored_similarity"].mean(),
        "mean_clip_similarity_improvement": group_df["clip_similarity_improvement"].mean(),
        "median_clip_similarity_improvement": group_df["clip_similarity_improvement"].median(),
        "mean_dinov2_damaged_similarity": group_df["dinov2_damaged_similarity"].mean(),
        "mean_dinov2_restored_similarity": group_df["dinov2_restored_similarity"].mean(),
        "mean_dinov2_similarity_improvement": group_df["dinov2_similarity_improvement"].mean(),
        "median_dinov2_similarity_improvement": group_df["dinov2_similarity_improvement"].median(),
        "mean_similarity_improvement": group_df["mean_similarity_improvement"].mean(),
        "median_mean_similarity_improvement": group_df["mean_similarity_improvement"].median(),
        "mean_absolute_feature_model_improvement_gap": group_df["absolute_feature_model_improvement_gap"].mean(),
        "median_absolute_feature_model_improvement_gap": group_df["absolute_feature_model_improvement_gap"].median(),
        "clip_improved_count": int(group_df["clip_improved"].sum()),
        "dinov2_improved_count": int(group_df["dinov2_improved"].sum()),
        "both_models_improved_count": int(group_df["both_feature_models_improved"].sum()),
        "direction_disagreement_count": int(group_df["feature_model_direction_disagreement"].sum()),
    }

    record["clip_improvement_rate"] = safe_rate(record["clip_improved_count"], row_count)
    record["dinov2_improvement_rate"] = safe_rate(record["dinov2_improved_count"], row_count)
    record["both_models_improved_rate"] = safe_rate(record["both_models_improved_count"], row_count)
    record["direction_disagreement_rate"] = safe_rate(record["direction_disagreement_count"], row_count)

    for column in group_columns:
        record[column] = group_df[column].iloc[0]

    return record


def build_summary_scope(source_df: pd.DataFrame, summary_scope: str, group_columns: list[str]) -> pd.DataFrame:
    if not group_columns:
        return pd.DataFrame([summarize_feature_group(source_df, summary_scope, [])])

    available_columns = [column for column in group_columns if column in source_df.columns]
    if len(available_columns) != len(group_columns):
        return pd.DataFrame()

    rows = []
    for _, group_df in source_df.groupby(available_columns, dropna=False, sort=False):
        rows.append(summarize_feature_group(group_df, summary_scope, available_columns))
    return pd.DataFrame(rows)


summary_scope_specs = [
    ("overall", []),
    ("by_evaluation_region", ["evaluation_region"]),
    ("by_mask_type_and_region", ["mask_type", "evaluation_region"]),
    ("by_category_and_region", ["category", "evaluation_region"]),
    ("by_style_and_region", ["style", "evaluation_region"]),
    ("by_dataset_and_region", ["dataset_name", "evaluation_region"]),
    ("by_model_and_region", ["model_name", "evaluation_region"]),
    ("by_zero_control_and_region", ["is_zero_control", "evaluation_region"]),
    ("by_direction_label_and_region", ["feature_direction_label", "evaluation_region"]),
]

summary_parts = [
    build_summary_scope(analysis_metrics_df, summary_scope, group_columns)
    for summary_scope, group_columns in summary_scope_specs
]
batch4_summary_df = pd.concat(
    [part for part in summary_parts if not part.empty],
    ignore_index=True,
    sort=False,
)

numeric_summary_columns = batch4_summary_df.select_dtypes(include=[np.number]).columns
batch4_summary_df[numeric_summary_columns] = batch4_summary_df[numeric_summary_columns].round(6)

summary_scope_order = {
    "overall": 0,
    "by_evaluation_region": 1,
    "by_mask_type_and_region": 2,
    "by_category_and_region": 3,
    "by_style_and_region": 4,
    "by_dataset_and_region": 5,
    "by_model_and_region": 6,
    "by_zero_control_and_region": 7,
    "by_direction_label_and_region": 8,
}
batch4_summary_df["_summary_scope_order"] = batch4_summary_df["summary_scope"].map(summary_scope_order).fillna(99)
batch4_summary_df = (
    batch4_summary_df
    .sort_values(["_summary_scope_order", "summary_scope", "feature_case_count", "summary_key"], ascending=[True, True, False, True])
    .drop(columns=["_summary_scope_order"])
    .reset_index(drop=True)
)

batch4_summary_df.to_csv(BATCH4_SUMMARY_PATH, index=False)

print(f"Saved summary CSV: {rel(BATCH4_SUMMARY_PATH)}")
print(f"Summary rows: {len(batch4_summary_df):,}")
display(batch4_summary_df.head(20))

Saved summary CSV: outputs/25_stable_diffusion_feature_similarity/analysis/stable_diffusion_feature_similarity_summary.csv
Summary rows: 90


,summary_scope,summary_key,group_columns,row_count,feature_case_count,unique_paintings,mean_clip_damaged_similarity,mean_clip_restored_similarity,mean_clip_similarity_improvement,median_clip_similarity_improvement,...,both_models_improved_rate,direction_disagreement_rate,evaluation_region,mask_type,category,style,dataset_name,model_name,is_zero_control,feature_direction_label
0,overall,all_cases,,2724,945,50,0.912477,0.963251,0.050773,0.053264,...,0.546990,0.073789,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,by_evaluation_region,content_region,evaluation_region,945,945,50,0.919176,0.966834,0.047658,0.051447,...,0.522751,0.071958,content_region,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,by_evaluation_region,full_image,evaluation_region,945,945,50,0.928282,0.970459,0.042177,0.045342,...,0.531217,0.064550,full_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,by_evaluation_region,mask_bbox_crop,evaluation_region,834,834,50,0.886980,0.951023,0.064043,0.065896,...,0.592326,0.086331,mask_bbox_crop,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,by_mask_type_and_region,loss_large | content_region,mask_type|evaluation_region,190,190,50,0.845438,0.962000,0.116562,0.116357,...,0.763158,0.236842,content_region,loss_large,NaN,NaN,NaN,NaN,NaN,NaN
5,by_mask_type_and_region,loss_large | full_image,mask_type|evaluation_region,190,190,50,0.874266,0.968176,0.093909,0.092951,...,0.773684,0.221053,full_image,loss_large,NaN,NaN,NaN,NaN,NaN,NaN
6,by_mask_type_and_region,loss_large | mask_bbox_crop,mask_type|evaluation_region,190,190,50,0.745935,0.918207,0.172272,0.169040,...,0.752632,0.242105,mask_bbox_crop,loss_large,NaN,NaN,NaN,NaN,NaN,NaN
7,by_mask_type_and_region,loss_small | content_region,mask_type|evaluation_region,143,143,50,0.889058,0.982231,0.093173,0.094522,...,0.951049,0.048951,content_region,loss_small,NaN,NaN,NaN,NaN,NaN,NaN
8,by_mask_type_and_region,loss_small | full_image,mask_type|evaluation_region,143,143,50,0.900362,0.986429,0.086066,0.086948,...,0.965035,0.034965,full_image,loss_small,NaN,NaN,NaN,NaN,NaN,NaN
9,by_mask_type_and_region,loss_small | mask_bbox_crop,mask_type|evaluation_region,143,143,50,0.892833,0.980552,0.087720,0.081161,...,0.930070,0.069930,mask_bbox_crop,loss_small,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Batch 4 / Cell 4 - Build selected cases CSV
selected_case_frames = []

preferred_selection_regions = [
    region for region in ["mask_bbox_crop", "content_region", "full_image"]
    if region in set(analysis_metrics_df["evaluation_region"].astype(str))
]

primary_selection_region = (
    "mask_bbox_crop"
    if "mask_bbox_crop" in preferred_selection_regions
    else preferred_selection_regions[0]
)

primary_region_df = analysis_metrics_df.loc[
    analysis_metrics_df["evaluation_region"].eq(primary_selection_region)
].copy()

def add_selected_cases(selection_reason: str, selected_df: pd.DataFrame) -> None:
    if selected_df.empty:
        return
    selected_copy = selected_df.copy()
    selected_copy.insert(0, "selection_reason", selection_reason)
    selected_case_frames.append(selected_copy)


add_selected_cases(
    f"top_mean_similarity_improvement_{primary_selection_region}",
    primary_region_df.sort_values(
        ["mean_similarity_improvement", "feature_case_id"],
        ascending=[False, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

add_selected_cases(
    f"bottom_mean_similarity_improvement_{primary_selection_region}",
    primary_region_df.sort_values(
        ["mean_similarity_improvement", "feature_case_id"],
        ascending=[True, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

add_selected_cases(
    f"largest_clip_dinov2_disagreement_gap_{primary_selection_region}",
    primary_region_df.sort_values(
        ["absolute_feature_model_improvement_gap", "feature_case_id"],
        ascending=[False, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

add_selected_cases(
    f"clip_improved_dinov2_not_{primary_selection_region}",
    primary_region_df.loc[
        primary_region_df["feature_direction_label"].eq("clip_improved_dinov2_not")
    ].sort_values(
        ["absolute_feature_model_improvement_gap", "feature_case_id"],
        ascending=[False, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

add_selected_cases(
    f"dinov2_improved_clip_not_{primary_selection_region}",
    primary_region_df.loc[
        primary_region_df["feature_direction_label"].eq("dinov2_improved_clip_not")
    ].sort_values(
        ["absolute_feature_model_improvement_gap", "feature_case_id"],
        ascending=[False, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

zero_control_df = analysis_metrics_df.loc[
    bool_series(analysis_metrics_df.get("is_zero_control", pd.Series(False, index=analysis_metrics_df.index)))
    & analysis_metrics_df["evaluation_region"].eq("content_region")
].copy()
add_selected_cases(
    "zero_control_content_region_reference",
    zero_control_df.sort_values(
        ["absolute_feature_model_improvement_gap", "feature_case_id"],
        ascending=[False, True],
        kind="stable",
    ).head(BATCH4_SELECTION_TOP_N),
)

if "mask_has_positive_pixels_for_feature_similarity" in analysis_metrics_df.columns:
    empty_nonzero_mask_df = analysis_metrics_df.loc[
        (~bool_series(analysis_metrics_df.get("is_zero_control", pd.Series(False, index=analysis_metrics_df.index))))
        & (~bool_series(analysis_metrics_df["mask_has_positive_pixels_for_feature_similarity"]))
        & analysis_metrics_df["evaluation_region"].eq("content_region")
    ].copy()
    add_selected_cases(
        "empty_nonzero_mask_content_region_reference",
        empty_nonzero_mask_df.sort_values(
            ["absolute_feature_model_improvement_gap", "feature_case_id"],
            ascending=[False, True],
            kind="stable",
        ).head(BATCH4_SELECTION_TOP_N),
    )

if selected_case_frames:
    batch4_selected_cases_df = pd.concat(selected_case_frames, ignore_index=True, sort=False)
else:
    batch4_selected_cases_df = pd.DataFrame()

selected_case_columns = [
    "selection_reason",
    "feature_case_id",
    "candidate_id",
    "restoration_case_id",
    "case_id",
    "dataset_name",
    "painting_id",
    "category",
    "title",
    "artist",
    "style",
    "model_name",
    "prompt_variant",
    "prompt_strategy",
    "mask_id",
    "mask_type",
    "evaluation_region",
    "region_pixel_count",
    "clip_damaged_similarity",
    "clip_restored_similarity",
    "clip_similarity_improvement",
    "dinov2_damaged_similarity",
    "dinov2_restored_similarity",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
    "clip_improved",
    "dinov2_improved",
    "both_feature_models_improved",
    "feature_model_direction_disagreement",
    "feature_direction_label",
    "absolute_feature_model_improvement_gap",
    "mean_similarity_rank_desc_by_region",
    "mean_similarity_rank_asc_by_region",
    "disagreement_gap_rank_desc_by_region",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "is_zero_control",
    "mask_has_positive_pixels_for_feature_similarity",
    "expected_feature_regions",
]
selected_case_columns = [
    column for column in selected_case_columns
    if column in batch4_selected_cases_df.columns
]

batch4_selected_cases_df = (
    batch4_selected_cases_df[selected_case_columns]
    .drop_duplicates(subset=["selection_reason", "feature_case_id", "evaluation_region"])
    .sort_values(["selection_reason", "feature_case_id"], kind="stable")
    .reset_index(drop=True)
)

batch4_selected_cases_df.to_csv(BATCH4_SELECTED_CASES_PATH, index=False)

print(f"Saved selected cases CSV: {rel(BATCH4_SELECTED_CASES_PATH)}")
print(f"Selected rows: {len(batch4_selected_cases_df):,}")
print("Selection reasons:")
print(batch4_selected_cases_df["selection_reason"].value_counts(dropna=False).to_string())

display(batch4_selected_cases_df.head(30))

Saved selected cases CSV: outputs/25_stable_diffusion_feature_similarity/analysis/stable_diffusion_feature_similarity_selected_cases.csv
Selected rows: 60
Selection reasons:
selection_reason
bottom_mean_similarity_improvement_mask_bbox_crop      12
clip_improved_dinov2_not_mask_bbox_crop                12
largest_clip_dinov2_disagreement_gap_mask_bbox_crop    12
top_mean_similarity_improvement_mask_bbox_crop         12
zero_control_content_region_reference                  12


,selection_reason,feature_case_id,candidate_id,restoration_case_id,case_id,dataset_name,painting_id,category,title,artist,...,mean_similarity_rank_desc_by_region,mean_similarity_rank_asc_by_region,disagreement_gap_rank_desc_by_region,clean_path,damaged_path,restored_path,mask_path,is_zero_control,mask_has_positive_pixels_for_feature_similarity,expected_feature_regions
0,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0240,sd__can__p028__loss_large__p04_full_context__s...,sd__can__p028__loss_large__p04_full_context__s...,p028_loss_large,canonical,p028,architecture_structured,Trading posts of the Dutch East India Company ...,workshop of Johannes Vinckboons,...,833,2,1,data/processed/clean/p028_clean.png,data/processed/masked/p028_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p028_loss_large_mask.png,True,True,full_image|content_region|mask_bbox_crop
1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0678,sd__syn__p001__discolouration__p00_generic__s2...,sd__syn__p001__discolouration__p00_generic__s2...,p001__discolouration__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,830,5,203,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|content_region|mask_bbox_crop
2,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0679,sd__syn__p001__discolouration__p01_style_perio...,sd__syn__p001__discolouration__p01_style_perio...,p001__discolouration__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,834,1,464,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|content_region|mask_bbox_crop
3,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0697,sd__syn__p001__partial_transparency__p00_gener...,sd__syn__p001__partial_transparency__p00_gener...,p001__partial_transparency__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,828,7,719,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|content_region|mask_bbox_crop
4,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0698,sd__syn__p001__partial_transparency__p01_style...,sd__syn__p001__partial_transparency__p01_style...,p001__partial_transparency__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,832,3,556,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|content_region|mask_bbox_crop
5,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0701,sd__syn__p001__partial_transparency__p04_full_...,sd__syn__p001__partial_transparency__p04_full_...,p001__partial_transparency__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,823,12,320,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|content_region|mask_bbox_crop
6,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0710,sd__syn__p001__water_stain__p01_style_period__...,sd__syn__p001__water_stain__p01_style_period__...,p001__water_stain__severe,synthetic_degradation,p001,portrait_figure,Juan de Pareja,Diego Velázquez,...,827,8,12,data/processed/clean/p001_clean.png,data/processed/degraded/synthetic_degradation/...,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...,True,True,full_image|cont

In [23]:
# Batch 4 / Cell 5 - Batch 4 validation and stage manifest update
required_summary_scopes = {
    "overall",
    "by_evaluation_region",
    "by_mask_type_and_region",
    "by_category_and_region",
    "by_dataset_and_region",
    "by_model_and_region",
    "by_zero_control_and_region",
    "by_direction_label_and_region",
}
actual_summary_scopes = set(batch4_summary_df["summary_scope"].dropna().astype(str))
missing_summary_scopes = sorted(required_summary_scopes - actual_summary_scopes)

actual_region_counts = analysis_metrics_df["evaluation_region"].value_counts().to_dict()
metric_error_rows = int(feature_metrics_df["status"].astype(str).eq("error").sum())
selected_reason_count = int(batch4_selected_cases_df["selection_reason"].nunique()) if not batch4_selected_cases_df.empty else 0

overall_summary_rows = batch4_summary_df.loc[
    batch4_summary_df["summary_scope"].eq("overall")
].copy()
overall_summary_case_count = (
    int(overall_summary_rows["feature_case_count"].iloc[0])
    if len(overall_summary_rows) == 1
    else -1
)

batch4_validation_rows = [
    validation_row(
        "batch3_validation_passed",
        validation_file_passed(BATCH3_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH3_VALIDATION_PATH),
        "Batch 3 validation did not pass.",
    ),
    validation_row(
        "batch3_metrics_loaded",
        len(feature_metrics_df),
        EXPECTED_FEATURE_ROWS,
        len(feature_metrics_df) == EXPECTED_FEATURE_ROWS,
        "Batch 3 metrics row count does not match expected feature rows.",
    ),
    validation_row(
        "batch4_no_metric_error_rows",
        metric_error_rows,
        0,
        metric_error_rows == 0,
        "Batch 4 input contains error metric rows.",
    ),
    validation_row(
        "batch4_region_counts_match_contract",
        actual_region_counts,
        EXPECTED_REGION_COUNTS,
        actual_region_counts == EXPECTED_REGION_COUNTS,
        "Batch 4 region counts do not match expected contract.",
    ),
    validation_row(
        "batch4_summary_written",
        rel(BATCH4_SUMMARY_PATH),
        "file exists",
        BATCH4_SUMMARY_PATH.is_file(),
        "Batch 4 summary CSV was not written.",
    ),
    validation_row(
        "batch4_selected_cases_written",
        rel(BATCH4_SELECTED_CASES_PATH),
        "file exists",
        BATCH4_SELECTED_CASES_PATH.is_file(),
        "Batch 4 selected cases CSV was not written.",
    ),
    validation_row(
        "batch4_summary_not_empty",
        len(batch4_summary_df),
        "> 0",
        len(batch4_summary_df) > 0,
        "Batch 4 summary is empty.",
    ),
    validation_row(
        "batch4_selected_cases_not_empty",
        len(batch4_selected_cases_df),
        "> 0",
        len(batch4_selected_cases_df) > 0,
        "Batch 4 selected cases table is empty.",
    ),
    validation_row(
        "batch4_required_summary_scopes_present",
        missing_summary_scopes,
        [],
        len(missing_summary_scopes) == 0,
        "Batch 4 summary is missing required scopes.",
    ),
    validation_row(
        "batch4_overall_case_count_matches_contract",
        overall_summary_case_count,
        EXPECTED_CANDIDATE_ROWS,
        overall_summary_case_count == EXPECTED_CANDIDATE_ROWS,
        "Overall summary feature-case count does not match expected candidate rows.",
    ),
    validation_row(
        "batch4_direction_labels_present",
        sorted(analysis_metrics_df["feature_direction_label"].dropna().unique().tolist()),
        "at least one direction label",
        analysis_metrics_df["feature_direction_label"].notna().any(),
        "No feature direction labels were created.",
    ),
    validation_row(
        "batch4_disagreement_diagnostics_present",
        int(analysis_metrics_df["absolute_feature_model_improvement_gap"].notna().sum()),
        len(analysis_metrics_df),
        int(analysis_metrics_df["absolute_feature_model_improvement_gap"].notna().sum()) == len(analysis_metrics_df),
        "Some disagreement diagnostic gaps are missing.",
    ),
    validation_row(
        "batch4_selection_reason_count",
        selected_reason_count,
        ">= 3",
        selected_reason_count >= 3,
        "Too few selected-case reason groups were produced.",
    ),
]

batch4_validation_df = pd.DataFrame(batch4_validation_rows)
batch4_validation_df.to_csv(BATCH4_VALIDATION_PATH, index=False)

batch4_passed = bool_series(batch4_validation_df["passed"]).all()

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch4_summaries_rankings_selected_cases"
stage_manifest["stage_status"] = "passed" if batch4_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch4"] = {
    "status": "passed" if batch4_passed else "failed",
    "summary": rel(BATCH4_SUMMARY_PATH),
    "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
    "validation": rel(BATCH4_VALIDATION_PATH),
    "input_metrics": rel(BATCH3_METRICS_PATH),
    "metric_rows": int(len(feature_metrics_df)),
    "ok_metric_rows": int(len(analysis_metrics_df)),
    "summary_rows": int(len(batch4_summary_df)),
    "selected_case_rows": int(len(batch4_selected_cases_df)),
    "selected_reason_count": selected_reason_count,
    "region_counts": actual_region_counts,
    "direction_label_counts": analysis_metrics_df["feature_direction_label"].value_counts(dropna=False).to_dict(),
    "checks_passed": int(bool_series(batch4_validation_df["passed"]).sum()),
    "checks_total": int(len(batch4_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved Batch 4 validation: {rel(BATCH4_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 4 checks passed: {int(bool_series(batch4_validation_df['passed']).sum())} / {len(batch4_validation_df)}")

display(batch4_validation_df)

if not batch4_passed:
    display(batch4_validation_df.loc[~bool_series(batch4_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 4 validation failed. Fix summaries/selected cases before Batch 5.")

print("Batch 4 passed. Continue to Batch 5.")

Saved Batch 4 validation: outputs/25_stable_diffusion_feature_similarity/validation/batch4_analysis_validation.csv
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 4 checks passed: 13 / 13


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch3_validation_passed,True,True,True,,2026-08-08T18:03:21+00:00
1,batch3_metrics_loaded,2724,2724,True,,2026-08-08T18:03:21+00:00
2,batch4_no_metric_error_rows,0,0,True,,2026-08-08T18:03:21+00:00
3,batch4_region_counts_match_contract,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,,2026-08-08T18:03:21+00:00
4,batch4_summary_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T18:03:21+00:00
5,batch4_selected_cases_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T18:03:21+00:00
6,batch4_summary_not_empty,90,> 0,True,,2026-08-08T18:03:21+00:00
7,batch4_selected_cases_not_empty,60,> 0,True,,2026-08-08T18:03:21+00:00
8,batch4_required_summary_scopes_present,[],[],True,,2026-08-08T18:03:21+00:00
9,batch4_overall_case_count_matches_contract,945,945,True,,2026-08-08T18:03:21+00:00


Batch 4 passed. Continue to Batch 5.


In [24]:
# Batch 5 / Cell 1 - Paths, imports, and inputs
import json
import math
import re
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps

BATCH5_AGGREGATE_FIGURES_DIR = globals().get(
    "BATCH5_AGGREGATE_FIGURES_DIR",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_aggregate_plots",
)
BATCH5_SELECTED_PANEL_DIR = globals().get(
    "BATCH5_SELECTED_PANEL_DIR",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_selected_case_panels",
)
BATCH5_FIGURE_MANIFEST_PATH = globals().get(
    "BATCH5_FIGURE_MANIFEST_PATH",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_figure_manifest.csv",
)
BATCH5_VALIDATION_PATH = globals().get(
    "BATCH5_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_figure_validation.csv",
)

BATCH3_METRICS_PATH = globals().get(
    "BATCH3_METRICS_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_metrics.csv",
)
BATCH4_SUMMARY_PATH = globals().get(
    "BATCH4_SUMMARY_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_summary.csv",
)
BATCH4_SELECTED_CASES_PATH = globals().get(
    "BATCH4_SELECTED_CASES_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_selected_cases.csv",
)
BATCH4_VALIDATION_PATH = globals().get(
    "BATCH4_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv",
)

BATCH5_PANEL_TILE_SIZE = tuple(globals().get("BATCH5_PANEL_TILE_SIZE", (250, 210)))
BATCH5_PANEL_PROGRESS_EVERY = int(globals().get("BATCH5_PANEL_PROGRESS_EVERY", 10))
BATCH5_DPI = int(globals().get("BATCH5_DPI", 160))

for directory in [BATCH5_AGGREGATE_FIGURES_DIR, BATCH5_SELECTED_PANEL_DIR, BATCH5_FIGURE_MANIFEST_PATH.parent, BATCH5_VALIDATION_PATH.parent]:
    Path(directory).mkdir(parents=True, exist_ok=True)

if not validation_file_passed(BATCH4_VALIDATION_PATH):
    raise RuntimeError("Batch 4 validation has not passed. Do not run Batch 5 yet.")

for required_path in [BATCH3_METRICS_PATH, BATCH4_SUMMARY_PATH, BATCH4_SELECTED_CASES_PATH]:
    if not Path(required_path).is_file():
        raise FileNotFoundError(f"Missing required Batch 5 input: {required_path}")

batch5_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
batch5_summary_df = pd.read_csv(BATCH4_SUMMARY_PATH)
batch5_selected_cases_df = pd.read_csv(BATCH4_SELECTED_CASES_PATH)

for column in [
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
    "absolute_feature_model_improvement_gap",
]:
    if column in batch5_metrics_df.columns:
        batch5_metrics_df[column] = pd.to_numeric(batch5_metrics_df[column], errors="coerce")
    if column in batch5_selected_cases_df.columns:
        batch5_selected_cases_df[column] = pd.to_numeric(batch5_selected_cases_df[column], errors="coerce")

batch5_plot_metrics_df = batch5_metrics_df.loc[
    batch5_metrics_df["status"].astype(str).eq("ok")
].copy()

batch5_plot_metrics_df["clip_improved"] = batch5_plot_metrics_df["clip_similarity_improvement"].gt(0)
batch5_plot_metrics_df["dinov2_improved"] = batch5_plot_metrics_df["dinov2_similarity_improvement"].gt(0)
batch5_plot_metrics_df["feature_direction_label"] = np.select(
    [
        batch5_plot_metrics_df["clip_improved"] & batch5_plot_metrics_df["dinov2_improved"],
        batch5_plot_metrics_df["clip_improved"] & ~batch5_plot_metrics_df["dinov2_improved"],
        ~batch5_plot_metrics_df["clip_improved"] & batch5_plot_metrics_df["dinov2_improved"],
    ],
    ["both_improved", "clip_improved_dinov2_not", "dinov2_improved_clip_not"],
    default="neither_improved",
)
batch5_plot_metrics_df["absolute_feature_model_improvement_gap"] = (
    batch5_plot_metrics_df["clip_similarity_improvement"]
    - batch5_plot_metrics_df["dinov2_similarity_improvement"]
).abs()

batch5_figure_manifest_records = []

print(f"Batch 5 metrics rows: {len(batch5_metrics_df):,}")
print(f"Batch 5 OK metrics rows: {len(batch5_plot_metrics_df):,}")
print(f"Batch 5 selected cases: {len(batch5_selected_cases_df):,}")
print(f"Aggregate plot dir: {rel(BATCH5_AGGREGATE_FIGURES_DIR)}")
print(f"Selected panel dir: {rel(BATCH5_SELECTED_PANEL_DIR)}")

Batch 5 metrics rows: 2,724
Batch 5 OK metrics rows: 2,724
Batch 5 selected cases: 60
Aggregate plot dir: outputs/25_stable_diffusion_feature_similarity/figures/stable_diffusion_feature_similarity_aggregate_plots
Selected panel dir: outputs/25_stable_diffusion_feature_similarity/figures/stable_diffusion_feature_similarity_selected_case_panels


In [25]:
# Batch 5 / Cell 2 - Figure utilities
def batch5_slug(value: object, fallback: str = "item") -> str:
    text_value = "" if pd.isna(value) else str(value).strip().lower()
    text_value = re.sub(r"[^a-z0-9]+", "_", text_value).strip("_")
    return text_value or fallback


def batch5_resolve_path(value: object) -> Path | None:
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_ROOT / path


def batch5_image_size(path: Path) -> tuple[int, int]:
    with Image.open(path) as image:
        return image.size


def batch5_record_figure(
    figure_id: str,
    figure_type: str,
    title: str,
    figure_path: Path,
    status: str = "ok",
    issue: str = "",
    source_rows: int | str = "",
    selection_reason: str = "",
    feature_case_id: str = "",
    evaluation_region: str = "",
) -> None:
    figure_path = Path(figure_path)
    file_exists = figure_path.is_file()
    width, height = (0, 0)
    if file_exists:
        try:
            width, height = batch5_image_size(figure_path)
        except Exception:
            width, height = (0, 0)

    batch5_figure_manifest_records.append(
        {
            "figure_id": figure_id,
            "figure_type": figure_type,
            "title": title,
            "path": rel(figure_path),
            "file_exists": bool(file_exists),
            "size_bytes": int(figure_path.stat().st_size) if file_exists else 0,
            "width_px": int(width),
            "height_px": int(height),
            "status": status,
            "issue": issue,
            "source_rows": source_rows,
            "selection_reason": selection_reason,
            "feature_case_id": feature_case_id,
            "evaluation_region": evaluation_region,
        }
    )


def batch5_save_plot(fig, figure_id: str, title: str, source_rows: int | str = "") -> Path:
    output_path = BATCH5_AGGREGATE_FIGURES_DIR / f"{figure_id}.png"
    fig.savefig(output_path, dpi=BATCH5_DPI, bbox_inches="tight")
    plt.close(fig)
    batch5_record_figure(
        figure_id=figure_id,
        figure_type="aggregate_plot",
        title=title,
        figure_path=output_path,
        source_rows=source_rows,
    )
    return output_path


def batch5_format_number(value: object, digits: int = 4) -> str:
    value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"

In [26]:
# Batch 5 / Cell 3 - Aggregate plot PNGs
plt.style.use("default")

region_order = ["full_image", "content_region", "mask_bbox_crop"]
region_order = [region for region in region_order if region in set(batch5_plot_metrics_df["evaluation_region"])]

region_summary_df = (
    batch5_plot_metrics_df
    .groupby("evaluation_region", dropna=False)
    .agg(
        mean_clip_improvement=("clip_similarity_improvement", "mean"),
        mean_dinov2_improvement=("dinov2_similarity_improvement", "mean"),
        mean_feature_improvement=("mean_similarity_improvement", "mean"),
        rows=("feature_case_id", "count"),
    )
    .reindex(region_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(region_summary_df))
width = 0.28
ax.bar(x - width, region_summary_df["mean_clip_improvement"], width, label="CLIP")
ax.bar(x, region_summary_df["mean_dinov2_improvement"], width, label="DINOv2")
ax.bar(x + width, region_summary_df["mean_feature_improvement"], width, label="Mean")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(region_summary_df["evaluation_region"], rotation=20, ha="right")
ax.set_ylabel("Mean similarity improvement")
ax.set_title("Mean feature-similarity improvement by evaluation region")
ax.legend()
batch5_save_plot(fig, "fs_plot_001_region_mean_improvement", "Mean feature improvement by region", len(batch5_plot_metrics_df))

fig, ax = plt.subplots(figsize=(7, 7))
for region, region_df in batch5_plot_metrics_df.groupby("evaluation_region", sort=False):
    ax.scatter(
        region_df["clip_similarity_improvement"],
        region_df["dinov2_similarity_improvement"],
        s=18,
        alpha=0.65,
        label=str(region),
    )
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("CLIP similarity improvement")
ax.set_ylabel("DINOv2 similarity improvement")
ax.set_title("CLIP vs DINOv2 improvement agreement")
ax.legend()
batch5_save_plot(fig, "fs_plot_002_clip_dinov2_scatter", "CLIP vs DINOv2 improvement scatter", len(batch5_plot_metrics_df))

fig, ax = plt.subplots(figsize=(9, 5))
for region, region_df in batch5_plot_metrics_df.groupby("evaluation_region", sort=False):
    ax.hist(
        region_df["mean_similarity_improvement"].dropna(),
        bins=30,
        alpha=0.45,
        label=str(region),
    )
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean similarity improvement")
ax.set_ylabel("Metric rows")
ax.set_title("Distribution of mean feature-similarity improvement")
ax.legend()
batch5_save_plot(fig, "fs_plot_003_improvement_histogram", "Mean improvement distribution", len(batch5_plot_metrics_df))

direction_counts_df = (
    batch5_plot_metrics_df
    .groupby(["evaluation_region", "feature_direction_label"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reindex(region_order)
)
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(direction_counts_df))
for label in direction_counts_df.columns:
    values = direction_counts_df[label].to_numpy()
    ax.bar(direction_counts_df.index, values, bottom=bottom, label=label)
    bottom += values
ax.set_ylabel("Metric rows")
ax.set_title("Feature-model direction labels by region")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=8)
batch5_save_plot(fig, "fs_plot_004_direction_counts", "Direction labels by region", len(batch5_plot_metrics_df))

selected_plot_df = batch5_selected_cases_df.copy()
if "mean_similarity_improvement" in selected_plot_df.columns and not selected_plot_df.empty:
    selected_plot_df = (
        selected_plot_df
        .sort_values("mean_similarity_improvement", ascending=False, kind="stable")
        .head(24)
        .copy()
    )
    selected_plot_df["plot_label"] = selected_plot_df.apply(
        lambda row: f"{str(row.get('feature_case_id', 'case'))[:24]} | {row.get('evaluation_region', '')}",
        axis=1,
    )
    fig, ax = plt.subplots(figsize=(10, max(5, 0.28 * len(selected_plot_df))))
    ax.barh(selected_plot_df["plot_label"], selected_plot_df["mean_similarity_improvement"])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("Mean similarity improvement")
    ax.set_title("Top selected cases by mean feature improvement")
    batch5_save_plot(fig, "fs_plot_005_selected_cases_top", "Top selected cases by mean improvement", len(selected_plot_df))

print("Aggregate plot manifest rows so far:", len(batch5_figure_manifest_records))

Aggregate plot manifest rows so far: 5


In [27]:
# Batch 5 / Cell 4 - Selected-case panel PNGs
def batch5_load_panel_image(path_value: object, role: str) -> Image.Image:
    path = batch5_resolve_path(path_value)
    if path is None or not path.is_file():
        raise FileNotFoundError(f"Missing {role} image path: {path_value}")

    with Image.open(path) as image:
        if role == "mask":
            image = image.convert("L")
            image = ImageOps.autocontrast(image)
            return image.convert("RGB")
        return image.convert("RGB")


def batch5_labeled_tile(image: Image.Image, label: str, tile_size: tuple[int, int]) -> Image.Image:
    tile_width, tile_height = tile_size
    label_height = 30
    canvas = Image.new("RGB", (tile_width, tile_height + label_height), "white")

    image_copy = ImageOps.contain(image, (tile_width, tile_height), method=Image.Resampling.LANCZOS)
    offset = ((tile_width - image_copy.width) // 2, label_height + (tile_height - image_copy.height) // 2)
    canvas.paste(image_copy, offset)

    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 0, tile_width, label_height], fill=(245, 245, 245))
    draw.text((8, 8), label, fill=(20, 20, 20))
    return canvas


def batch5_metric_card(row: pd.Series, size: tuple[int, int]) -> Image.Image:
    width, height = size
    card = Image.new("RGB", (width, height), (250, 250, 250))
    draw = ImageDraw.Draw(card)

    lines = [
        f"Selection: {row.get('selection_reason', '')}",
        f"Feature case: {row.get('feature_case_id', '')}",
        f"Region: {row.get('evaluation_region', '')}",
        f"Mask: {row.get('mask_type', '')}",
        "",
        f"CLIP delta: {batch5_format_number(row.get('clip_similarity_improvement'))}",
        f"DINOv2 delta: {batch5_format_number(row.get('dinov2_similarity_improvement'))}",
        f"Mean delta: {batch5_format_number(row.get('mean_similarity_improvement'))}",
        f"Gap: {batch5_format_number(row.get('absolute_feature_model_improvement_gap'))}",
        f"Direction: {row.get('feature_direction_label', '')}",
    ]

    y = 10
    for line in lines:
        wrapped = textwrap.wrap(str(line), width=38) or [""]
        for wrapped_line in wrapped:
            draw.text((10, y), wrapped_line, fill=(20, 20, 20))
            y += 15
        y += 2

    return card


def batch5_create_selected_case_panel(row: pd.Series, output_path: Path) -> None:
    tile_size = BATCH5_PANEL_TILE_SIZE
    card_size = (330, tile_size[1] + 30)

    clean_tile = batch5_labeled_tile(batch5_load_panel_image(row.get("clean_path"), "clean"), "Clean", tile_size)
    damaged_tile = batch5_labeled_tile(batch5_load_panel_image(row.get("damaged_path"), "damaged"), "Damaged", tile_size)
    mask_tile = batch5_labeled_tile(batch5_load_panel_image(row.get("mask_path"), "mask"), "Mask", tile_size)
    restored_tile = batch5_labeled_tile(batch5_load_panel_image(row.get("restored_path"), "restored"), "Restored", tile_size)
    card_tile = batch5_metric_card(row, card_size)

    panel_width = clean_tile.width + damaged_tile.width + mask_tile.width + restored_tile.width + card_tile.width
    panel_height = max(clean_tile.height, card_tile.height)

    panel = Image.new("RGB", (panel_width, panel_height), "white")
    x_offset = 0
    for tile in [clean_tile, damaged_tile, mask_tile, restored_tile, card_tile]:
        panel.paste(tile, (x_offset, 0))
        x_offset += tile.width

    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel.save(output_path)


panel_source_df = (
    batch5_selected_cases_df
    .drop_duplicates(subset=["selection_reason", "feature_case_id", "evaluation_region"])
    .reset_index(drop=True)
)

for idx, (_, row) in enumerate(panel_source_df.iterrows(), start=1):
    figure_id = f"fs_panel_{idx:03d}"
    output_path = BATCH5_SELECTED_PANEL_DIR / f"{figure_id}.png"
    status = "ok"
    issue = ""

    try:
        batch5_create_selected_case_panel(row, output_path)
    except Exception as exc:
        status = "error"
        issue = str(exc)

    batch5_record_figure(
        figure_id=figure_id,
        figure_type="selected_case_panel",
        title="Selected feature-similarity case panel",
        figure_path=output_path,
        status=status,
        issue=issue,
        source_rows=1,
        selection_reason=str(row.get("selection_reason", "")),
        feature_case_id=str(row.get("feature_case_id", "")),
        evaluation_region=str(row.get("evaluation_region", "")),
    )

    if idx % BATCH5_PANEL_PROGRESS_EVERY == 0:
        print(f"Generated selected-case panels: {idx}/{len(panel_source_df)}")

batch5_figure_manifest_df = pd.DataFrame(batch5_figure_manifest_records)
batch5_figure_manifest_df.to_csv(BATCH5_FIGURE_MANIFEST_PATH, index=False)

print(f"Saved figure manifest: {rel(BATCH5_FIGURE_MANIFEST_PATH)}")
print("Figure type counts:")
print(batch5_figure_manifest_df["figure_type"].value_counts(dropna=False).to_string())
display(batch5_figure_manifest_df.head(20))

Generated selected-case panels: 10/60
Generated selected-case panels: 20/60
Generated selected-case panels: 30/60
Generated selected-case panels: 40/60
Generated selected-case panels: 50/60
Generated selected-case panels: 60/60
Saved figure manifest: outputs/25_stable_diffusion_feature_similarity/figures/stable_diffusion_feature_similarity_figure_manifest.csv
Figure type counts:
figure_type
selected_case_panel    60
aggregate_plot          5


,figure_id,figure_type,title,path,file_exists,size_bytes,width_px,height_px,status,issue,source_rows,selection_reason,feature_case_id,evaluation_region
0,fs_plot_001_region_mean_improvement,aggregate_plot,Mean feature improvement by region,outputs/25_stable_diffusion_feature_similarity...,True,52625,1243,780,ok,,2724,,,
1,fs_plot_002_clip_dinov2_scatter,aggregate_plot,CLIP vs DINOv2 improvement scatter,outputs/25_stable_diffusion_feature_similarity...,True,216268,999,997,ok,,2724,,,
2,fs_plot_003_improvement_histogram,aggregate_plot,Mean improvement distribution,outputs/25_stable_diffusion_feature_similarity...,True,49604,1236,751,ok,,2724,,,
3,fs_plot_004_direction_counts,aggregate_plot,Direction labels by region,outputs/25_stable_diffusion_feature_similarity...,True,49398,1360,780,ok,,2724,,,
4,fs_plot_005_selected_cases_top,aggregate_plot,Top selected cases by mean improvement,outputs/25_stable_diffusion_feature_similarity...,True,159217,1609,963,ok,,24,,,
5,fs_panel_001,selected_case_panel,Selected feature-similarity case panel,outputs/25_stable_diffusion_feature_similarity...,True,75600,1330,240,ok,,1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0240,mask_bbox_crop
6,fs_panel_002,selected_case_panel,Selected feature-similarity case panel,outputs/25_stable_diffusion_feature_similarity...,True,146286,1330,240,ok,,1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0678,mask_bbox_crop
7,fs_panel_003,selected_case_panel,Selected feature-similarity case panel,outputs/25_stable_diffusion_feature_similarity...,True,146320,1330,240,ok,,1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0679,mask_bbox_crop
8,fs_panel_004,selected_case_panel,Selected feature-similarity case panel,outputs/25_stable_diffusion_feature_similarity...,True,165591,1330,240,ok,,1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0697,mask_bbox_crop
9,fs_panel_005,selected_case_panel,Selected feature-similarity case panel,outputs/25_stable_diffusion_feature_similarity...,True,167961,1330,240,ok,,1,bottom_mean_similarity_improvement_mask_bbox_crop,sd24_0698,mask_bbox_crop


In [28]:
# Batch 5 / Cell 5 - Figure validation and stage manifest
figure_type_counts = batch5_figure_manifest_df["figure_type"].value_counts().to_dict()
error_rows = int(batch5_figure_manifest_df["status"].ne("ok").sum())
missing_files = int((~bool_series(batch5_figure_manifest_df["file_exists"])).sum())
empty_files = int(pd.to_numeric(batch5_figure_manifest_df["size_bytes"], errors="coerce").fillna(0).le(0).sum())

expected_panel_rows = int(len(panel_source_df))
actual_panel_rows = int(figure_type_counts.get("selected_case_panel", 0))
actual_plot_rows = int(figure_type_counts.get("aggregate_plot", 0))

batch5_validation_rows = [
    validation_row(
        "batch4_validation_passed",
        validation_file_passed(BATCH4_VALIDATION_PATH),
        True,
        validation_file_passed(BATCH4_VALIDATION_PATH),
        "Batch 4 validation did not pass.",
    ),
    validation_row(
        "aggregate_plot_count",
        actual_plot_rows,
        ">= 5",
        actual_plot_rows >= 5,
        "Expected at least five aggregate plot PNGs.",
    ),
    validation_row(
        "selected_panel_count",
        actual_panel_rows,
        expected_panel_rows,
        actual_panel_rows == expected_panel_rows and expected_panel_rows > 0,
        "Selected-case panel count does not match selected-case input rows.",
    ),
    validation_row(
        "figure_manifest_written",
        rel(BATCH5_FIGURE_MANIFEST_PATH),
        "file exists",
        BATCH5_FIGURE_MANIFEST_PATH.is_file(),
        "Figure manifest CSV was not written.",
    ),
    validation_row(
        "figure_manifest_not_empty",
        len(batch5_figure_manifest_df),
        "> 0",
        len(batch5_figure_manifest_df) > 0,
        "Figure manifest is empty.",
    ),
    validation_row(
        "figure_rows_all_ok",
        error_rows,
        0,
        error_rows == 0,
        "One or more figure rows have error status.",
    ),
    validation_row(
        "figure_files_all_exist",
        missing_files,
        0,
        missing_files == 0,
        "One or more figure files are missing.",
    ),
    validation_row(
        "figure_files_nonempty",
        empty_files,
        0,
        empty_files == 0,
        "One or more figure files are empty.",
    ),
    validation_row(
        "figure_dimensions_present",
        int((batch5_figure_manifest_df["width_px"].gt(0) & batch5_figure_manifest_df["height_px"].gt(0)).sum()),
        len(batch5_figure_manifest_df),
        bool((batch5_figure_manifest_df["width_px"].gt(0) & batch5_figure_manifest_df["height_px"].gt(0)).all()),
        "One or more figure files have invalid image dimensions.",
    ),
]

batch5_validation_df = pd.DataFrame(batch5_validation_rows)
batch5_validation_df.to_csv(BATCH5_VALIDATION_PATH, index=False)

batch5_passed = bool_series(batch5_validation_df["passed"]).all()

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch5_aggregate_plots_and_selected_case_panels"
stage_manifest["stage_status"] = "passed" if batch5_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch5"] = {
    "status": "passed" if batch5_passed else "failed",
    "aggregate_figures_dir": rel(BATCH5_AGGREGATE_FIGURES_DIR),
    "selected_panel_dir": rel(BATCH5_SELECTED_PANEL_DIR),
    "figure_manifest": rel(BATCH5_FIGURE_MANIFEST_PATH),
    "validation": rel(BATCH5_VALIDATION_PATH),
    "figure_rows": int(len(batch5_figure_manifest_df)),
    "figure_type_counts": figure_type_counts,
    "aggregate_plot_count": actual_plot_rows,
    "selected_panel_count": actual_panel_rows,
    "error_rows": error_rows,
    "missing_files": missing_files,
    "empty_files": empty_files,
    "checks_passed": int(bool_series(batch5_validation_df["passed"]).sum()),
    "checks_total": int(len(batch5_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved Batch 5 validation: {rel(BATCH5_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 5 checks passed: {int(bool_series(batch5_validation_df['passed']).sum())} / {len(batch5_validation_df)}")

display(batch5_validation_df)

if not batch5_passed:
    display(batch5_validation_df.loc[~bool_series(batch5_validation_df["passed"]), ["check_name", "failure_message"]])
    display(batch5_figure_manifest_df.loc[batch5_figure_manifest_df["status"].ne("ok") | ~bool_series(batch5_figure_manifest_df["file_exists"])])
    raise RuntimeError("Batch 5 validation failed. Fix figure generation before final closeout.")

print("Batch 5 passed. Feature-similarity figures and selected-case panels are ready.")

Saved Batch 5 validation: outputs/25_stable_diffusion_feature_similarity/validation/batch5_figure_validation.csv
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 5 checks passed: 9 / 9


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch4_validation_passed,True,True,True,,2026-08-08T18:17:26+00:00
1,aggregate_plot_count,5,>= 5,True,,2026-08-08T18:17:26+00:00
2,selected_panel_count,60,60,True,,2026-08-08T18:17:26+00:00
3,figure_manifest_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T18:17:26+00:00
4,figure_manifest_not_empty,65,> 0,True,,2026-08-08T18:17:26+00:00
5,figure_rows_all_ok,0,0,True,,2026-08-08T18:17:26+00:00
6,figure_files_all_exist,0,0,True,,2026-08-08T18:17:26+00:00
7,figure_files_nonempty,0,0,True,,2026-08-08T18:17:26+00:00
8,figure_dimensions_present,65,65,True,,2026-08-08T18:17:26+00:00


Batch 5 passed. Feature-similarity figures and selected-case panels are ready.


In [29]:
# Batch 6 / Cell 1 - Paths, imports, and inputs
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd()))

STAGE_MANIFEST_PATH = Path(globals().get(
    "STAGE_MANIFEST_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_stage_manifest.json",
))

BATCH6_ARTIFACT_INDEX_PATH = Path(globals().get(
    "BATCH6_ARTIFACT_INDEX_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_artifact_index.csv",
))
BATCH6_VALIDATION_PATH = Path(globals().get(
    "BATCH6_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_final_validation.csv",
))
BATCH6_HANDOFF_MANIFEST_PATH = Path(globals().get(
    "BATCH6_HANDOFF_MANIFEST_PATH",
    STAGE_MANIFEST_PATH.parent / "stable_diffusion_feature_similarity_handoff_manifest.json",
))

BATCH3_METRICS_PATH = Path(globals().get(
    "BATCH3_METRICS_PATH",
    OUTPUT_DIRS["metrics"] / "stable_diffusion_feature_similarity_metrics.csv",
))
BATCH4_SUMMARY_PATH = Path(globals().get(
    "BATCH4_SUMMARY_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_summary.csv",
))
BATCH4_SELECTED_CASES_PATH = Path(globals().get(
    "BATCH4_SELECTED_CASES_PATH",
    OUTPUT_DIRS["analysis"] / "stable_diffusion_feature_similarity_selected_cases.csv",
))
BATCH4_VALIDATION_PATH = Path(globals().get(
    "BATCH4_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv",
))
BATCH5_FIGURE_MANIFEST_PATH = Path(globals().get(
    "BATCH5_FIGURE_MANIFEST_PATH",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_figure_manifest.csv",
))
BATCH5_VALIDATION_PATH = Path(globals().get(
    "BATCH5_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_feature_similarity_figure_validation.csv",
))
BATCH5_AGGREGATE_FIGURES_DIR = Path(globals().get(
    "BATCH5_AGGREGATE_FIGURES_DIR",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_aggregate_plots",
))
BATCH5_SELECTED_PANEL_DIR = Path(globals().get(
    "BATCH5_SELECTED_PANEL_DIR",
    OUTPUT_DIRS["figures"] / "stable_diffusion_feature_similarity_selected_case_panels",
))

for directory in [
    BATCH6_ARTIFACT_INDEX_PATH.parent,
    BATCH6_VALIDATION_PATH.parent,
    BATCH6_HANDOFF_MANIFEST_PATH.parent,
]:
    directory.mkdir(parents=True, exist_ok=True)

if not validation_file_passed(BATCH5_VALIDATION_PATH):
    raise RuntimeError("Batch 5 validation has not passed. Do not run Batch 6 yet.")

required_batch6_inputs = [
    BATCH3_METRICS_PATH,
    BATCH4_SUMMARY_PATH,
    BATCH4_SELECTED_CASES_PATH,
    BATCH4_VALIDATION_PATH,
    BATCH5_FIGURE_MANIFEST_PATH,
    BATCH5_VALIDATION_PATH,
]

missing_batch6_inputs = [path for path in required_batch6_inputs if not path.exists()]
if missing_batch6_inputs:
    raise FileNotFoundError(f"Missing required Batch 6 inputs: {missing_batch6_inputs}")

batch6_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
batch6_summary_df = pd.read_csv(BATCH4_SUMMARY_PATH)
batch6_selected_cases_df = pd.read_csv(BATCH4_SELECTED_CASES_PATH)
batch6_figure_manifest_df = pd.read_csv(BATCH5_FIGURE_MANIFEST_PATH)

print(f"Loaded metrics rows: {len(batch6_metrics_df):,}")
print(f"Loaded summary rows: {len(batch6_summary_df):,}")
print(f"Loaded selected-case rows: {len(batch6_selected_cases_df):,}")
print(f"Loaded figure manifest rows: {len(batch6_figure_manifest_df):,}")

Loaded metrics rows: 2,724
Loaded summary rows: 90
Loaded selected-case rows: 60
Loaded figure manifest rows: 65


In [30]:
# Batch 6 / Cell 2 - Final utility functions
def batch6_rel(path: Path) -> str:
    try:
        return rel(path)
    except Exception:
        try:
            return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
        except Exception:
            return str(path)


def batch6_utc_now_iso() -> str:
    if "utc_now_iso" in globals():
        return utc_now_iso()
    return datetime.now(timezone.utc).isoformat()


def batch6_read_json_if_exists(path: Path) -> dict:
    if "read_json_if_exists" in globals():
        return read_json_if_exists(path)
    path = Path(path)
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def batch6_json_safe(value):
    if "to_json_safe" in globals():
        return to_json_safe(value)
    if isinstance(value, Path):
        return batch6_rel(value)
    if isinstance(value, dict):
        return {str(key): batch6_json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [batch6_json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [batch6_json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if not isinstance(value, (dict, list, tuple, Path)) else False:
        return None
    return value


def batch6_bool_series(series: pd.Series) -> pd.Series:
    if "bool_series" in globals():
        return bool_series(series)
    return series.astype(str).str.lower().isin(["true", "1", "yes", "passed", "ok"])


def batch6_validation_row(check_name, actual, expected, passed, failure_message):
    if "validation_row" in globals():
        return validation_row(check_name, actual, expected, passed, failure_message)
    return {
        "check_name": check_name,
        "actual": json.dumps(batch6_json_safe(actual), ensure_ascii=False),
        "expected": json.dumps(batch6_json_safe(expected), ensure_ascii=False),
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


def batch6_validation_file_passed(path: Path) -> bool:
    path = Path(path)
    if not path.is_file():
        return False
    if "validation_file_passed" in globals():
        return bool(validation_file_passed(path))

    validation_df = pd.read_csv(path)
    if "passed" not in validation_df.columns or validation_df.empty:
        return False
    return bool(batch6_bool_series(validation_df["passed"]).all())


def batch6_csv_shape(path: Path) -> tuple[int | None, int | None]:
    try:
        df = pd.read_csv(path)
        return int(len(df)), int(len(df.columns))
    except Exception:
        return None, None


def batch6_directory_file_count(path: Path, suffix: str | None = None) -> int:
    path = Path(path)
    if not path.is_dir():
        return 0
    files = path.rglob("*")
    if suffix is not None:
        files = (file for file in files if file.is_file() and file.suffix.lower() == suffix.lower())
    else:
        files = (file for file in files if file.is_file())
    return sum(1 for _ in files)


def batch6_path_size_bytes(path: Path) -> int:
    path = Path(path)
    if path.is_file():
        return int(path.stat().st_size)
    if path.is_dir():
        return int(sum(file.stat().st_size for file in path.rglob("*") if file.is_file()))
    return 0

In [31]:
# Batch 6 / Cell 3 - Build artifact index
def batch6_artifact_spec(batch, artifact_role, path, required=True, artifact_type="file", notes=""):
    return {
        "batch": batch,
        "artifact_role": artifact_role,
        "path": Path(path) if path is not None else None,
        "required": bool(required),
        "artifact_type": artifact_type,
        "notes": notes,
    }


batch6_artifact_specs = [
    batch6_artifact_spec("batch3", "feature_similarity_metrics", BATCH3_METRICS_PATH, True, "csv"),
    batch6_artifact_spec("batch4", "summary_csv", BATCH4_SUMMARY_PATH, True, "csv"),
    batch6_artifact_spec("batch4", "selected_cases_csv", BATCH4_SELECTED_CASES_PATH, True, "csv"),
    batch6_artifact_spec("batch4", "validation_csv", BATCH4_VALIDATION_PATH, True, "validation_csv"),
    batch6_artifact_spec("batch5", "figure_manifest_csv", BATCH5_FIGURE_MANIFEST_PATH, True, "csv"),
    batch6_artifact_spec("batch5", "validation_csv", BATCH5_VALIDATION_PATH, True, "validation_csv"),
    batch6_artifact_spec("batch5", "aggregate_figure_directory", BATCH5_AGGREGATE_FIGURES_DIR, True, "directory"),
    batch6_artifact_spec("batch5", "selected_case_panel_directory", BATCH5_SELECTED_PANEL_DIR, True, "directory"),
    batch6_artifact_spec("stage", "stage_manifest_json", STAGE_MANIFEST_PATH, False, "json"),
]

for optional_global_name, batch_name, role_name in [
    ("BATCH1_INPUT_CASES_PATH", "batch1", "input_cases_csv"),
    ("BATCH1_VALIDATION_PATH", "batch1", "validation_csv"),
    ("BATCH2_INPUT_CASES_PATH", "batch2", "input_cases_csv"),
    ("BATCH2_EMBEDDING_MANIFEST_PATH", "batch2", "embedding_manifest_csv"),
    ("BATCH2_VALIDATION_PATH", "batch2", "validation_csv"),
    ("BATCH3_VALIDATION_PATH", "batch3", "validation_csv"),
]:
    if optional_global_name in globals():
        artifact_type = "validation_csv" if "VALIDATION" in optional_global_name else "csv"
        batch6_artifact_specs.append(
            batch6_artifact_spec(
                batch_name,
                role_name,
                Path(globals()[optional_global_name]),
                True,
                artifact_type,
                f"Discovered from notebook global: {optional_global_name}",
            )
        )

artifact_rows = []
for index, spec in enumerate(batch6_artifact_specs, start=1):
    path = spec["path"]
    exists = bool(path.exists()) if path is not None else False
    is_file = bool(path.is_file()) if exists else False
    is_dir = bool(path.is_dir()) if exists else False
    size_bytes = batch6_path_size_bytes(path) if exists else 0
    csv_rows, csv_columns = batch6_csv_shape(path) if exists and is_file and path.suffix.lower() == ".csv" else (None, None)
    png_count = batch6_directory_file_count(path, ".png") if exists and is_dir else 0

    status = "ok"
    issue = ""
    if spec["required"] and not exists:
        status = "missing"
        issue = "Required artifact does not exist."
    elif spec["required"] and size_bytes <= 0:
        status = "empty"
        issue = "Required artifact exists but is empty."
    elif spec["artifact_type"] == "directory" and spec["required"] and png_count <= 0:
        status = "empty_directory"
        issue = "Required directory has no PNG files."

    artifact_rows.append(
        {
            "artifact_id": f"{index:03d}_{spec['batch']}_{spec['artifact_role']}",
            "batch": spec["batch"],
            "artifact_role": spec["artifact_role"],
            "artifact_type": spec["artifact_type"],
            "path": batch6_rel(path) if path is not None else "",
            "required": spec["required"],
            "file_exists": exists,
            "is_file": is_file,
            "is_directory": is_dir,
            "size_bytes": size_bytes,
            "csv_rows": csv_rows,
            "csv_columns": csv_columns,
            "png_count": png_count,
            "status": status,
            "issue": issue,
            "notes": spec["notes"],
        }
    )

batch6_artifact_index_df = pd.DataFrame(artifact_rows)
batch6_artifact_index_df = batch6_artifact_index_df.sort_values(
    ["batch", "artifact_role", "artifact_id"],
    kind="stable",
).reset_index(drop=True)

batch6_artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_PATH, index=False)

print(f"Saved artifact index: {batch6_rel(BATCH6_ARTIFACT_INDEX_PATH)}")
print("Artifact status counts:")
print(batch6_artifact_index_df["status"].value_counts(dropna=False).to_string())
display(batch6_artifact_index_df)

Saved artifact index: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_artifact_index.csv
Artifact status counts:
status
ok    13


,artifact_id,batch,artifact_role,artifact_type,path,required,file_exists,is_file,is_directory,size_bytes,csv_rows,csv_columns,png_count,status,issue,notes
0,010_batch1_input_cases_csv,batch1,input_cases_csv,csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,3981426,945.0,336.0,0,ok,,Discovered from notebook global: BATCH1_INPUT_...
1,011_batch1_validation_csv,batch1,validation_csv,validation_csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,2144,23.0,6.0,0,ok,,Discovered from notebook global: BATCH1_VALIDA...
2,012_batch2_validation_csv,batch2,validation_csv,validation_csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,2229,19.0,6.0,0,ok,,Discovered from notebook global: BATCH2_VALIDA...
3,001_batch3_feature_similarity_metrics,batch3,feature_similarity_metrics,csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,2467756,2724.0,59.0,0,ok,,
4,013_batch3_validation_csv,batch3,validation_csv,validation_csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,3678,31.0,6.0,0,ok,,Discovered from notebook global: BATCH3_VALIDA...
5,003_batch4_selected_cases_csv,batch4,selected_cases_csv,csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,58616,60.0,39.0,0,ok,,
6,002_batch4_summary_csv,batch4,summary_csv,csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,25433,90.0,34.0,0,ok,,
7,004_batch4_validation_csv,batch4,validation_csv,validation_csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,1441,13.0,6.0,0,ok,,
8,007_batch5_aggregate_figure_directory,batch5,aggregate_figure_directory,directory,outputs/25_stable_diffusion_feature_similarity...,True,True,False,True,527112,NaN,NaN,5,ok,,
9,005_batch5_figure_manifest_csv,batch5,figure_manifest_csv,csv,outputs/25_stable_diffusion_feature_similarity...,True,True,True,False,19358,65.0,14.0,0,ok,,


In [32]:
# Batch 6 / Cell 4 - Final validation and handoff manifest
batch6_required_artifacts_df = batch6_artifact_index_df.loc[
    batch6_bool_series(batch6_artifact_index_df["required"])
].copy()

batch6_validation_artifacts_df = batch6_artifact_index_df.loc[
    batch6_artifact_index_df["artifact_type"].eq("validation_csv")
    & batch6_bool_series(batch6_artifact_index_df["file_exists"])
].copy()

previous_validation_results = []
for _, artifact_row in batch6_validation_artifacts_df.iterrows():
    validation_path = PROJECT_ROOT / artifact_row["path"]
    if not validation_path.exists():
        validation_path = Path(artifact_row["path"])
    previous_validation_results.append(
        {
            "artifact_id": artifact_row["artifact_id"],
            "path": artifact_row["path"],
            "passed": batch6_validation_file_passed(validation_path),
        }
    )

previous_validation_results_df = pd.DataFrame(previous_validation_results)

figure_manifest_ok_df = batch6_figure_manifest_df.copy()
if "file_exists" in figure_manifest_ok_df.columns:
    figure_files_exist = batch6_bool_series(figure_manifest_ok_df["file_exists"]).all()
else:
    figure_files_exist = False

figure_status_ok = (
    figure_manifest_ok_df["status"].astype(str).eq("ok").all()
    if "status" in figure_manifest_ok_df.columns and not figure_manifest_ok_df.empty
    else False
)

aggregate_plot_count = int(
    figure_manifest_ok_df["figure_type"].astype(str).eq("aggregate_plot").sum()
    if "figure_type" in figure_manifest_ok_df.columns
    else 0
)
selected_panel_count = int(
    figure_manifest_ok_df["figure_type"].astype(str).eq("selected_case_panel").sum()
    if "figure_type" in figure_manifest_ok_df.columns
    else 0
)

metric_ok_rows = int(batch6_metrics_df["status"].astype(str).eq("ok").sum()) if "status" in batch6_metrics_df.columns else len(batch6_metrics_df)
metric_error_rows = int(batch6_metrics_df["status"].astype(str).eq("error").sum()) if "status" in batch6_metrics_df.columns else 0

expected_feature_rows = globals().get("EXPECTED_FEATURE_ROWS", None)
expected_candidate_rows = globals().get("EXPECTED_CANDIDATE_ROWS", None)

batch6_validation_rows = [
    batch6_validation_row(
        "batch5_validation_passed",
        batch6_validation_file_passed(BATCH5_VALIDATION_PATH),
        True,
        batch6_validation_file_passed(BATCH5_VALIDATION_PATH),
        "Batch 5 validation has not passed.",
    ),
    batch6_validation_row(
        "artifact_index_written",
        batch6_rel(BATCH6_ARTIFACT_INDEX_PATH),
        "file exists",
        BATCH6_ARTIFACT_INDEX_PATH.is_file(),
        "Artifact index CSV was not written.",
    ),
    batch6_validation_row(
        "required_artifacts_exist",
        int((~batch6_bool_series(batch6_required_artifacts_df["file_exists"])).sum()),
        0,
        int((~batch6_bool_series(batch6_required_artifacts_df["file_exists"])).sum()) == 0,
        "One or more required artifacts are missing.",
    ),
    batch6_validation_row(
        "required_artifacts_nonempty",
        int(pd.to_numeric(batch6_required_artifacts_df["size_bytes"], errors="coerce").fillna(0).le(0).sum()),
        0,
        int(pd.to_numeric(batch6_required_artifacts_df["size_bytes"], errors="coerce").fillna(0).le(0).sum()) == 0,
        "One or more required artifacts are empty.",
    ),
    batch6_validation_row(
        "previous_validation_files_passed",
        previous_validation_results_df.to_dict("records") if not previous_validation_results_df.empty else [],
        "all passed",
        bool(not previous_validation_results_df.empty and previous_validation_results_df["passed"].all()),
        "One or more previous validation files failed or could not be read.",
    ),
    batch6_validation_row(
        "metrics_error_rows_zero",
        metric_error_rows,
        0,
        metric_error_rows == 0,
        "Feature-similarity metrics contain error rows.",
    ),
    batch6_validation_row(
        "summary_not_empty",
        len(batch6_summary_df),
        "> 0",
        len(batch6_summary_df) > 0,
        "Batch 4 summary is empty.",
    ),
    batch6_validation_row(
        "selected_cases_not_empty",
        len(batch6_selected_cases_df),
        "> 0",
        len(batch6_selected_cases_df) > 0,
        "Batch 4 selected cases are empty.",
    ),
    batch6_validation_row(
        "figure_manifest_not_empty",
        len(batch6_figure_manifest_df),
        "> 0",
        len(batch6_figure_manifest_df) > 0,
        "Batch 5 figure manifest is empty.",
    ),
    batch6_validation_row(
        "figure_manifest_status_ok",
        figure_status_ok,
        True,
        figure_status_ok,
        "One or more Batch 5 figure manifest rows are not ok.",
    ),
    batch6_validation_row(
        "figure_manifest_files_exist",
        figure_files_exist,
        True,
        bool(figure_files_exist),
        "One or more Batch 5 figure files are missing.",
    ),
    batch6_validation_row(
        "aggregate_plots_present",
        aggregate_plot_count,
        ">= 5",
        aggregate_plot_count >= 5,
        "Expected at least five aggregate plots.",
    ),
    batch6_validation_row(
        "selected_case_panels_match_selected_cases",
        selected_panel_count,
        len(batch6_selected_cases_df),
        selected_panel_count == len(batch6_selected_cases_df) and selected_panel_count > 0,
        "Selected-case panel count does not match selected-case rows.",
    ),
]

if expected_feature_rows is not None:
    batch6_validation_rows.append(
        batch6_validation_row(
            "metrics_row_count_matches_contract",
            len(batch6_metrics_df),
            int(expected_feature_rows),
            len(batch6_metrics_df) == int(expected_feature_rows),
            "Metrics row count does not match EXPECTED_FEATURE_ROWS.",
        )
    )

if expected_candidate_rows is not None:
    batch6_validation_rows.append(
        batch6_validation_row(
            "selected_cases_have_expected_source_pool_available",
            metric_ok_rows,
            f">= {int(expected_candidate_rows)}",
            metric_ok_rows >= int(expected_candidate_rows),
            "OK metric rows are fewer than expected candidate rows.",
        )
    )

batch6_validation_df = pd.DataFrame(batch6_validation_rows)
batch6_passed = bool(batch6_bool_series(batch6_validation_df["passed"]).all())

batch6_handoff_manifest = {
    "stage": "stable_diffusion_feature_similarity",
    "stage_status": "passed" if batch6_passed else "failed",
    "generated_at_utc": batch6_utc_now_iso(),
    "core_inputs": {
        "metrics": batch6_rel(BATCH3_METRICS_PATH),
        "summary": batch6_rel(BATCH4_SUMMARY_PATH),
        "selected_cases": batch6_rel(BATCH4_SELECTED_CASES_PATH),
        "figure_manifest": batch6_rel(BATCH5_FIGURE_MANIFEST_PATH),
    },
    "core_outputs": {
        "artifact_index": batch6_rel(BATCH6_ARTIFACT_INDEX_PATH),
        "final_validation": batch6_rel(BATCH6_VALIDATION_PATH),
        "handoff_manifest": batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH),
        "aggregate_figures_dir": batch6_rel(BATCH5_AGGREGATE_FIGURES_DIR),
        "selected_case_panel_dir": batch6_rel(BATCH5_SELECTED_PANEL_DIR),
    },
    "counts": {
        "metric_rows": int(len(batch6_metrics_df)),
        "metric_ok_rows": metric_ok_rows,
        "metric_error_rows": metric_error_rows,
        "summary_rows": int(len(batch6_summary_df)),
        "selected_case_rows": int(len(batch6_selected_cases_df)),
        "figure_manifest_rows": int(len(batch6_figure_manifest_df)),
        "aggregate_plot_count": aggregate_plot_count,
        "selected_panel_count": selected_panel_count,
        "artifact_index_rows": int(len(batch6_artifact_index_df)),
    },
    "validation": {
        "status": "passed" if batch6_passed else "failed",
        "checks_passed": int(batch6_bool_series(batch6_validation_df["passed"]).sum()),
        "checks_total": int(len(batch6_validation_df)),
        "previous_validation_results": previous_validation_results,
    },
    "handoff_notes": [
        "Use the summary CSV for aggregate interpretation.",
        "Use selected-case CSV plus selected-case panels for qualitative inspection.",
        "Use the figure manifest as the audit trail for every generated PNG.",
        "Feature-similarity plots are diagnostics, not final semantic proof.",
    ],
}

BATCH6_HANDOFF_MANIFEST_PATH.write_text(
    json.dumps(batch6_json_safe(batch6_handoff_manifest), indent=2),
    encoding="utf-8",
)

batch6_validation_df.to_csv(BATCH6_VALIDATION_PATH, index=False)

stage_manifest = batch6_read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest["stage"] = "batch6_final_validation_artifact_index_handoff"
stage_manifest["stage_status"] = "passed" if batch6_passed else "failed"
stage_manifest["updated_at_utc"] = batch6_utc_now_iso()
stage_manifest["batch6"] = {
    "status": "passed" if batch6_passed else "failed",
    "artifact_index": batch6_rel(BATCH6_ARTIFACT_INDEX_PATH),
    "final_validation": batch6_rel(BATCH6_VALIDATION_PATH),
    "handoff_manifest": batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH),
    "checks_passed": int(batch6_bool_series(batch6_validation_df["passed"]).sum()),
    "checks_total": int(len(batch6_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(
    json.dumps(batch6_json_safe(stage_manifest), indent=2),
    encoding="utf-8",
)

print(f"Saved final validation: {batch6_rel(BATCH6_VALIDATION_PATH)}")
print(f"Saved handoff manifest: {batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH)}")
print(f"Updated stage manifest: {batch6_rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 6 checks passed: {int(batch6_bool_series(batch6_validation_df['passed']).sum())} / {len(batch6_validation_df)}")

display(batch6_validation_df)

if not batch6_passed:
    display(batch6_validation_df.loc[~batch6_bool_series(batch6_validation_df["passed"]), ["check_name", "failure_message"]])
    display(batch6_artifact_index_df.loc[batch6_artifact_index_df["status"].ne("ok")])
    raise RuntimeError("Batch 6 validation failed. Fix the listed handoff issues before closing this stage.")

print("Batch 6 passed. Stable Diffusion feature-similarity stage is closed and ready for downstream thesis analysis.")

Saved final validation: outputs/25_stable_diffusion_feature_similarity/validation/stable_diffusion_feature_similarity_final_validation.csv
Saved handoff manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_handoff_manifest.json
Updated stage manifest: outputs/25_stable_diffusion_feature_similarity/manifests/stable_diffusion_feature_similarity_stage_manifest.json
Batch 6 checks passed: 15 / 15


,check_name,actual,expected,passed,failure_message,checked_at_utc
0,batch5_validation_passed,True,True,True,,2026-08-08T18:21:56+00:00
1,artifact_index_written,outputs/25_stable_diffusion_feature_similarity...,file exists,True,,2026-08-08T18:21:56+00:00
2,required_artifacts_exist,0,0,True,,2026-08-08T18:21:56+00:00
3,required_artifacts_nonempty,0,0,True,,2026-08-08T18:21:56+00:00
4,previous_validation_files_passed,"[{'artifact_id': '011_batch1_validation_csv', ...",all passed,True,,2026-08-08T18:21:56+00:00
5,metrics_error_rows_zero,0,0,True,,2026-08-08T18:21:56+00:00
6,summary_not_empty,90,> 0,True,,2026-08-08T18:21:56+00:00
7,selected_cases_not_empty,60,> 0,True,,2026-08-08T18:21:56+00:00
8,figure_manifest_not_empty,65,> 0,True,,2026-08-08T18:21:56+00:00
9,figure_manifest_status_ok,True,True,True,,2026-08-08T18:21:56+00:00


Batch 6 passed. Stable Diffusion feature-similarity stage is closed and ready for downstream thesis analysis.
